# IV-5 マイクロホンアレイ信号処理（ノートブック版）

教科書の IV-5 の 6 節のプログラムを、1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。8 節の到来方向の実験に数分〜10 分前後、全部で 5〜15 分ほどです。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 自前の実装（`common/array.py`）を pyroomacoustics と照合する | B | `check_array.py` |
| 2 | 空間相関行列の固有値と音源の数 | C | `scm_eig.py` |
| 3 | ビームパターンと空間エイリアシングを見て、聞く | D | `beampattern.py` |
| 4 | 遅延和・MVDR・MPDR・対角装荷を比べて、聞く | E | `beamform_demo.py` |
| 5 | 窓の長さと瞬時混合近似 | F | `narrowband.py` |
| 6 | 非同期録音のずれ（SFO）のシミュレーション | H | `sfo.py` |
| 7 | 拍手の時間差の解析を、合成の録音で試す | I の 16 | `clap_gcc.py --synthetic` |
| 8 | 到来方向の推定誤差を、部屋を単位にして比べる | G | `doa_experiment.py`・`doa_stats.py` |
| 9 | 確認問題の答え合わせ | J の 23 | `quiz_check.py` |

- 次のセルたちが、共通ライブラリ（`common/` の `__init__.py`・`stft.py`・`audio_io.py`・`plotting.py`（IV-2）と、この章の `array.py`）を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の各章の 6 節と同じなので、ほかのファイルは要りません。
- 各節のコードは 6 節のプログラムそのものです。変えたのは、ファイルの場所（`__file__` の代わりにノートブックの隣を使う）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、日本語のフォントの候補を足すこと、コマンドの引数を `sys.argv` で渡すこと、出力をいったん `cap` に受け取って次のセルで表示し、`ok()` で教科書の手順の「確かめること」を確かめることだけです。手順 B の 3（わざと符号を間違えてみる）は、`array.py` を書きかえる代わりに、小さなセルを足して同じことを確かめます。
- **入れていないもの**：手順 I の 17〜22（Mac とスマホの2台で拍手を録り、`clap_gcc.py data/mac.wav data/phone.wav` で解析する）は、Colab ではマイクで録れず、スマホの録音もないので入れていません（合成の録音で試す 16 だけ入れました）。手順 F の 12（第7章の `bss_experiment.py` の窓を変える）は第7章のプログラムを使うので、手順 J の 24（論文を読む）はコードがないので、入れていません。Mac のターミナルで教科書の手順どおりに行ってください。自分の録音を Colab で解析するときは、左の「ファイル」に `data/mac.wav`・`data/phone.wav` を上げて、7 節の `sys.argv` を書きかえます（画面の項目名は要確認）。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。保存した wav をそのままの大きさで鳴らすため、`IPython.display.Audio(..., normalize=False)` を使います。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を一番小さくしてください。
- 話し声は librosa の例の音源 `libri1`・`libri2` を使います（初回はダウンロードします。ネットワークが要ります）。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、pyroomacoustics・scipy の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・statsmodels・matplotlib・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"), ("pyroomacoustics", "pyroomacoustics"),
                 ("fast_bss_eval", "fast_bss_eval"), ("statsmodels", "statsmodels")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, pyroomacoustics, statsmodels
print("numpy", numpy.__version__, "scipy", scipy.__version__, "pyroomacoustics", pyroomacoustics.__version__, "statsmodels", statsmodels.__version__)
print("（教科書は pyroomacoustics 0.10.1・scipy 1.18.1 で確かめた）")
import os
for d in ["common", "data"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out

In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x

In [ ]:
%%writefile common/plotting.py
"""第IV部の共通ライブラリ：図の設定とスペクトログラムの表示（IV-2 で定義）

  import matplotlib; matplotlib.use("Agg")      # 図をファイルに保存するだけのとき（第7章と同じ）
  from common.plotting import specshow
  specshow(X, fs, S, ax=ax, title="…")          # X は stft の結果（複素数）か、振幅・パワー

日本語のフォントは既存の章と同じ（Hiragino Sans）。
"""
import numpy as np
import matplotlib

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

__all__ = ["specshow", "to_db"]


def to_db(X, power=False, ref=None, floor_db=-120.0):
    """X（複素数・振幅・パワー）を dB にする。振幅は 20log10、パワーは 10log10。
    ref を与えないと、最大値を 0 dB にする"""
    A = np.abs(X)
    if power:
        D = 10 * np.log10(A + 1e-20)
    else:
        D = 20 * np.log10(A + 1e-20)
    D -= (D.max() if ref is None else (10 if power else 20) * np.log10(ref))
    return np.maximum(D, floor_db)


def specshow(X, fs, S, ax=None, power=False, db_range=80.0, fmax=None, title=None, cmap="magma",
             colorbar=True):
    """スペクトログラムを表示する。X は (F, T)。横軸はフレームの窓の中心の時刻 t·S/fs [s]、
    縦軸はビンの周波数 k·fs/L [Hz]（L = 2(F−1)）。色は最大値から db_range [dB] 下までを表示する。"""
    import matplotlib.pyplot as plt
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("specshow に渡す X は (F, T)。多チャネルなら X[:, :, m] のように1つ選ぶ")
    F, T = X.shape
    L = 2 * (F - 1)
    D = to_db(X, power=power)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    df, dt = fs / L, S / fs
    # 各ビン・各フレームの値を、その中心の周りの長方形で塗る
    extent = (-dt / 2, (T - 1) * dt + dt / 2, -df / 2, (F - 1) * df + df / 2)
    im = ax.imshow(D, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-db_range, vmax=0, interpolation="nearest")
    ax.set_xlabel("時間 [s]")
    ax.set_ylabel("周波数 [Hz]")
    if fmax is not None:
        ax.set_ylim(0, fmax)
    if title:
        ax.set_title(title)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="dB（最大を 0）")
    return im

In [ ]:
%%writefile common/array.py
"""第IV部の共通ライブラリ：マイクロホンアレイ（IV-5 で定義。IV-6・最終演習K でも使う）

約束（IV-5 の本文の記号と同じ）
  mics      : マイクの位置。形 (D, M)（D = 2 か 3、単位 m）。pyroomacoustics の R と同じ並び
  azimuth   : 方位角 θ [rad]。x 軸から反時計回り（pyroomacoustics と同じ）。
              直線アレイを x 軸に沿って置くと、θ = 90° が正面（ブロードサイド）、0°・180° が軸の方向（エンドファイア）
  u(θ)      : 原点から音源の方向を向く単位ベクトル [cosθ, sinθ(, 0)]
  遅延      : 平面波（遠方場）では、マイク m に届く時刻は原点より τ_m = −u·r_m / c だけ遅い
  ステアリングベクトル a_f(θ) の成分 a_m = exp(−j2πf τ_m) = exp(+j2πf u·r_m / c)
              （STFT の e^{−j2πkm/L} の約束で「τ 遅れると e^{−j2πfτ} が掛かる」。pyroomacoustics.doa と同じ符号）
  X         : 観測の STFT。形 (F, T, M)（common.stft.stft の多チャネルの出力）
  重み w     : 出力 y_ft = w_fᴴ x_ft = Σ_m conj(w_fm) x_ftm。W の形は (F, M)

関数
  linear_array, circular_array, unit_vector, far_field_delays, steering_vector,
  scm, ds_weights, mvdr_weights, apply_weights, beam_pattern, white_noise_gain,
  gcc_phat, tdoa_to_azimuth, music_spectrum, srp_phat_spectrum,
  sfo_resample, sfo_phase
"""
import numpy as np

C_SOUND = 343.0          # 音速 [m/s]（約 20 ℃。pyroomacoustics の既定と同じ）

__all__ = ["C_SOUND", "linear_array", "circular_array", "unit_vector", "far_field_delays",
           "steering_vector", "scm", "ds_weights", "mvdr_weights", "apply_weights", "beam_pattern",
           "white_noise_gain", "gcc_phat", "tdoa_to_azimuth", "music_spectrum", "srp_phat_spectrum",
           "sfo_resample", "sfo_phase"]


# ---------------------------------------------------------------- 配置
def linear_array(M, d, center=(0.0, 0.0), angle=0.0):
    """間隔 d [m] の M 本の直線アレイ。中心が center、向きが angle [rad]（x 軸から）。形 (2, M)
    （pyroomacoustics.linear_2D_array と同じ）"""
    u = np.array([np.cos(angle), np.sin(angle)])
    k = np.arange(M) - (M - 1) / 2
    return np.asarray(center, float)[:, None] + d * k[None, :] * u[:, None]


def circular_array(M, radius, center=(0.0, 0.0), phi0=0.0):
    """半径 radius [m] の M 本の円形アレイ。形 (2, M)（pyroomacoustics.circular_2D_array と同じ）"""
    phi = phi0 + 2 * np.pi * np.arange(M) / M
    return np.asarray(center, float)[:, None] + radius * np.vstack([np.cos(phi), np.sin(phi)])


def unit_vector(azimuth, dim=2):
    """方位角（スカラーか (K,)）の単位ベクトル。形 (dim,) か (dim, K)"""
    az = np.asarray(azimuth, float)
    u = [np.cos(az), np.sin(az)] + ([np.zeros_like(az)] if dim == 3 else [])
    return np.array(u)


# ---------------------------------------------------------------- ステアリングベクトル
def far_field_delays(mics, azimuth, c=C_SOUND):
    """平面波の到来時刻の遅れ τ_m = −u·r_m / c [s]（原点を基準）。形 (M,) か (K, M)"""
    u = unit_vector(azimuth, dim=mics.shape[0])
    return -(u.T @ mics) / c


def steering_vector(mics, freqs, azimuth=None, source=None, c=C_SOUND, amplitude=False):
    """ステアリングベクトル。
    遠方場（azimuth を与える）：a_m = exp(−j2πf τ_m)、τ_m = −u·r_m / c。
        azimuth がスカラーなら形 (F, M)、(K,) なら (F, K, M)
    近距離場（source に音源の位置 (D,) を与える）：アレイの中心 c̄ からの距離 ρ_0 を基準に
        τ_m = (|p − r_m| − |p − c̄|) / c。amplitude=True なら距離の減衰 ρ_0/|p − r_m| も掛ける。形 (F, M)
    """
    f = np.atleast_1d(np.asarray(freqs, float))
    if (azimuth is None) == (source is None):
        raise ValueError("azimuth か source のどちらか一方を与える")
    if azimuth is not None:
        tau = far_field_delays(mics, azimuth, c)                      # (M,) か (K, M)
        return np.exp(-2j * np.pi * f.reshape((-1,) + (1,) * tau.ndim) * tau)
    p = np.asarray(source, float).reshape(-1, 1)[: mics.shape[0]]
    dist = np.linalg.norm(p - mics, axis=0)                          # (M,)
    ref = np.linalg.norm(p[:, 0] - mics.mean(axis=1))
    a = np.exp(-2j * np.pi * f[:, None] * (dist - ref)[None, :] / c)
    return a * (ref / dist)[None, :] if amplitude else a


# ---------------------------------------------------------------- 空間相関行列とビームフォーマ
def scm(X):
    """空間相関行列 R_f = (1/T) Σ_t x_ft x_ftᴴ（複素正規分布の共分散の最尤推定。IV-4-3-4）。
    X (F, T, M) → (F, M, M)"""
    return np.einsum("ftm,ftn->fmn", X, X.conj()) / X.shape[1]


def ds_weights(a):
    """遅延和ビームフォーマ w = a / M（目的の方向の利得が 1）。a (F, M) → (F, M)"""
    return a / a.shape[-1]


def mvdr_weights(a, R, loading=0.0):
    """MVDR（最小分散無歪み応答）w = R⁻¹a / (aᴴR⁻¹a)。
    loading > 0 なら対角装荷：R + ε I、ε = loading × tr(R)/M（周波数ごと）。a (F, M)、R (F, M, M)"""
    R = np.asarray(R, complex)
    M = R.shape[-1]
    if loading > 0:
        eps = loading * np.real(np.trace(R, axis1=-2, axis2=-1)) / M
        R = R + eps[:, None, None] * np.eye(M)[None]
    Ria = np.linalg.solve(R, a[..., None])[..., 0]                    # R⁻¹a
    den = np.einsum("fm,fm->f", a.conj(), Ria)                       # aᴴR⁻¹a（実数で正）
    return Ria / den[:, None]


def apply_weights(W, X):
    """y_ft = w_fᴴ x_ft。W (F, M)、X (F, T, M) → (F, T)"""
    return np.einsum("fm,ftm->ft", W.conj(), X)


def beam_pattern(W, mics, freqs, azimuths, c=C_SOUND):
    """ビームパターン B_f(θ) = w_fᴴ a_f(θ)（複素数）。W (F, M) → (F, K)"""
    A = steering_vector(mics, freqs, azimuth=np.asarray(azimuths, float), c=c)   # (F, K, M)
    return np.einsum("fm,fkm->fk", W.conj(), A)


def white_noise_gain(W, a):
    """白色雑音利得 WNG = |wᴴa|² / (wᴴw)（マイクごとに独立な雑音を何倍に抑えるか。DS で M）"""
    num = np.abs(np.einsum("fm,fm->f", W.conj(), a)) ** 2
    return num / np.real(np.einsum("fm,fm->f", W.conj(), W))


# ---------------------------------------------------------------- 到来方向推定
def gcc_phat(x1, x2, fs, max_tau=None, interp=1):
    """GCC-PHAT で、x2 が x1 より何秒遅れているか（τ > 0 なら x2 が遅い）を推定する。
    G(ω) = X2(ω) X1*(ω) / |X2(ω) X1*(ω)|、cc(τ) = IDFT[G]（長さを interp 倍にして細かくする）
    戻り値: tau [s], lags [s], cc（lags の順に並べた相互相関）"""
    x1 = np.asarray(x1, float)
    x2 = np.asarray(x2, float)
    n = len(x1) + len(x2)                                            # 円状のたたみ込みにならない長さ
    n += n % 2
    G = np.fft.rfft(x2, n) * np.conj(np.fft.rfft(x1, n))
    G /= np.abs(G) + 1e-20
    cc = np.fft.irfft(G, interp * n)
    half = interp * n // 2
    cc = np.concatenate([cc[-half:], cc[:half]])                     # ラグ −half 〜 half−1
    lags = (np.arange(-half, half)) / (interp * fs)
    if max_tau is not None:
        keep = np.abs(lags) <= max_tau
        cc, lags = cc[keep], lags[keep]
    return float(lags[np.argmax(cc)]), lags, cc


def tdoa_to_azimuth(tau, d, c=C_SOUND):
    """2本のマイク（x 軸上、マイク2 が +x 側に d）の時間差 τ（マイク2 の遅れ）から方位角 θ [rad]。
    τ = −d cosθ / c なので θ = arccos(−cτ/d)（0〜π。表と裏は区別できない）"""
    return float(np.arccos(np.clip(-c * tau / d, -1.0, 1.0)))


def music_spectrum(X, mics, freqs, azimuths, n_src=1, normalize=False, c=C_SOUND):
    """広帯域 MUSIC の空間スペクトル（周波数ごとの 1/|aᴴ E_n E_nᴴ a| の平均）。
    X (F', T, M) は使う周波数のビンだけを取り出した STFT、freqs (F',) はその周波数 [Hz]。
    normalize=True なら周波数ごとに最大を 1 にしてから平均（pyroomacoustics の NormMUSIC）"""
    R = scm(X)
    _, V = np.linalg.eigh(R)                                         # 固有値の小さい順
    En = V[:, :, : X.shape[2] - n_src]                               # 雑音部分空間 (F', M, M−N)
    A = steering_vector(mics, freqs, azimuth=azimuths, c=c)          # (F', K, M)
    proj = np.einsum("fkm,fmj->fkj", A.conj(), En)                   # aᴴ E_n
    P = 1.0 / np.sum(np.abs(proj) ** 2, axis=2)                      # (F', K)
    if normalize:
        P = P / P.max(axis=1, keepdims=True)
    return P.mean(axis=0)


def srp_phat_spectrum(X, mics, freqs, azimuths, c=C_SOUND):
    """SRP-PHAT：振幅を 1 にそろえた観測 x̃ = x/|x|（成分ごと）に遅延和をかけた出力のパワー
    |aᴴ x̃|² を、フレームと周波数で平均したもの。X (F', T, M)"""
    Xn = X / np.maximum(np.abs(X), 1e-14)
    A = steering_vector(mics, freqs, azimuth=azimuths, c=c)          # (F', K, M)
    out = np.einsum("fkm,ftm->fkt", A.conj(), Xn)
    return np.mean(np.abs(out) ** 2, axis=(0, 2))


# ---------------------------------------------------------------- 非同期録音（標本化周波数のずれ）
def sfo_resample(x, eps, pad=4096):
    """標本化周波数が fs(1+eps) の機器で録った信号を作る：y[n] = x_c(n / (fs(1+eps)))。
    FFT による帯域制限の補間（scipy.signal.resample）。eps はサンプル数の都合で少し丸める。
    戻り値: y（長さは x と同じ）, 実際に使った eps"""
    from scipy.signal import resample
    x = np.asarray(x, float)
    N = len(x) + pad                                                 # 端の回り込みを防ぐゼロ
    if eps != 0:                                                     # N·eps が整数に近くなる N を選ぶ（丸めの誤差を小さく）
        cand = np.arange(N, N + int(2 / abs(eps)) + 2)
        N = int(cand[np.argmin(np.abs(cand * eps - np.round(cand * eps)))])
        pad = N - len(x)
    num = int(round(N * (1 + eps)))
    y = resample(np.concatenate([x, np.zeros(pad)]), num)
    return y[: len(x)], num / N - 1


def sfo_phase(F, T, L, S, eps):
    """フレーム t（中心の時刻 tS/fs）での、ずれ eps による位相の回転 exp(−j2π k eps t S / L)。形 (F, T)。
    標本化周波数が fs(1+eps) のチャネルは、時刻 τ で eps·τ だけ遅れて見える（IV-5-3-10）。
    補正は、そのチャネルの STFT にこの共役を掛ける"""
    k = np.arange(F)[:, None]
    t = np.arange(T)[None, :]
    return np.exp(-2j * np.pi * k * eps * t * S / L)

In [ ]:
import sys, re, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
warnings.filterwarnings("ignore", message="Glyph")
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common import plotting                      # import すると Hiragino Sans が設定される
from common import array as ar
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
print("array.py を読み込めました：音速", ar.C_SOUND, "m/s")


def listen_file(path):
    """保存した wav をそのままの大きさで再生ボタンにする。normalize=False を消さない"""
    import numpy as np, soundfile   # 各節のプログラムが np などの名前を使うので、ここで読み直す
    x, fs = soundfile.read(path)
    assert np.max(np.abs(x)) <= 0.31, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1, text=None):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout if text is None else text)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが ok などの名前を使うことがあるので、確かめのセルで戻す

## 1. 自前の実装の照合（IV-5-3-2・3-4・3-5・3-6、手順 B、`check_array.py`）

`common/array.py` のステアリングベクトル・遅延和・MVDR・MUSIC・SRP-PHAT・GCC-PHAT を、pyroomacoustics と比べます（数秒）。最後に「すべて一致」と出れば合格です。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-5：common/array.py の自前の実装を、pyroomacoustics と数値で照合する

使い方: cd ~/home-lab/partIV/iv5 && uv run python check_array.py
  [1] 手計算の例（2本、d = 5.66 cm、θ = 60°、1 kHz）
  [2] ステアリングベクトル：pra.doa の ModeVector、pra.Beamformer の近距離場のベクトル
  [3] 遅延和の重みとビームパターン：pra.Beamformer.far_field_weights・response
  [4] MVDR：pra.Beamformer.rake_max_sinr_weights（周波数領域の MVDR と同じ式）
  [5] MUSIC・NormMUSIC・SRP-PHAT の空間スペクトル：pra.doa
  [6] GCC-PHAT：pra.experimental.tdoa
どれも許容誤差を決めて assert する。通れば最後に「すべて一致」と出る。
"""
import pathlib
import sys

import numpy as np
import pyroomacoustics as pra

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common.stft import stft                                         # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000
rng = np.random.default_rng(0)


class Src:
    """pra.Beamformer の rake_* が使う「音源の鏡像の位置」だけを持つ小さな入れ物（直接音だけ）"""
    def __init__(self, p):
        self.images = np.asarray(p, float).reshape(2, 1)


def report(name, diff, tol):
    ok = diff < tol
    print(f"  {name:44s} 最大の差 {diff:.2e}（許容 {tol:.0e}）{'OK' if ok else '★不一致'}")
    assert ok, name


def main():
    print("[1] 手計算の例：2本（x 軸上、間隔 5.66 cm）、θ = 60°、f = 1 kHz")
    mics2 = ar.linear_array(2, 0.0566)
    tau = ar.far_field_delays(mics2, np.radians(60))
    a = ar.steering_vector(mics2, [1000.0], azimuth=np.radians(60))[0]
    print(f"  τ = {tau * 1e6} µs、マイク2 のマイク1 に対する遅れ {(tau[1] - tau[0]) * 1e6:.2f} µs")
    print(f"  a / a_1 = {np.round(a / a[0], 4)}、位相差 {np.angle(a[1] / a[0]):.4f} rad")

    print("[2] ステアリングベクトル")
    mics = ar.circular_array(6, 0.05)
    nfft = 512
    grid = np.linspace(0, 2 * np.pi, 72, endpoint=False)
    mv = pra.doa.ModeVector(mics, FS, nfft, ar.C_SOUND, pra.doa.GridCircle(azimuth=grid))
    k = np.arange(nfft // 2 + 1)
    ours = ar.steering_vector(mics, k * FS / nfft, azimuth=grid)      # (F, K, M)
    theirs = np.transpose(mv[k, :, :], (0, 2, 1))                     # (F, M, K) → (F, K, M)
    report("遠方場：pra.doa.ModeVector", np.max(np.abs(ours - theirs)), 1e-9)
    bf = pra.Beamformer(mics, FS, N=nfft)
    p = np.array([1.2, 0.7])
    f = bf.frequencies
    pra_near = bf.steering_vector_2D_from_point(f, p, attn=False)     # (M, F) exp(−jω|p − r_m|/c)
    ref = np.linalg.norm(p - mics.mean(axis=1))
    ours = ar.steering_vector(mics, f, source=p) * np.exp(-2j * np.pi * f * ref / ar.C_SOUND)[:, None]
    report("近距離場：Beamformer.steering_vector_2D_from_point", np.max(np.abs(ours - pra_near.T)), 1e-9)

    print("[3] 遅延和の重みとビームパターン")
    lin = ar.linear_array(4, 0.0566)
    bf = pra.Beamformer(lin, FS, N=nfft)
    phi = np.radians(70)
    bf.far_field_weights(phi)                                         # exp(j2πf·proj/c)、proj から最大を引く
    a = ar.steering_vector(lin, bf.frequencies, azimuth=phi)
    w = ar.ds_weights(a)
    proj = ar.unit_vector(phi) @ (lin - lin.mean(axis=1, keepdims=True))
    shift = np.exp(-2j * np.pi * bf.frequencies * proj.max() / ar.C_SOUND)[:, None]
    report("重み：far_field_weights / M", np.max(np.abs(bf.weights.T / 4 - w * shift)), 1e-12)
    pra.constants.set("ffdist", 1e5)                                  # response の「遠方」を 100 km にする
    bf.weights = w.T.copy()
    angles = np.radians(np.arange(0, 360, 5.0))
    fq, resp = bf.response(angles, 2000.0)
    ours = ar.beam_pattern(w, lin, bf.frequencies, angles)[np.argmin(np.abs(bf.frequencies - 2000.0))]
    report(f"ビームパターン |B|（{fq:.0f} Hz）：response", np.max(np.abs(np.abs(ours) - np.abs(resp))), 1e-4)
    pra.constants.set("ffdist", 10.0)

    print("[4] MVDR（周波数領域）")
    bf = pra.Beamformer(lin, FS, N=nfft)
    s_pos, i_pos = np.array([0.3, 1.5]), np.array([-1.4, 0.6])
    Rn = 0.01 * np.eye(4)
    bf.rake_max_sinr_weights(Src(s_pos), Src(i_pos), R_n=Rn, attn=False)
    f = bf.frequencies
    a_s = ar.steering_vector(lin, f, source=s_pos)
    a_i = ar.steering_vector(lin, f, source=i_pos)
    R = a_i[:, :, None] * a_i[:, None, :].conj() + Rn[None]
    w = ar.mvdr_weights(a_s, R)
    ref = np.linalg.norm(s_pos - lin.mean(axis=1))
    w_pra = bf.weights.T * np.exp(2j * np.pi * f * ref / ar.C_SOUND)[:, None]   # 基準の位相をそろえる
    report("rake_max_sinr_weights", np.max(np.abs(w - w_pra)[1:]), 1e-8)
    g_s = np.abs(np.einsum("fm,fm->f", w.conj(), a_s))
    g_i = np.abs(np.einsum("fm,fm->f", w.conj(), a_i))
    k1, k3 = np.argmin(np.abs(f - 1000)), np.argmin(np.abs(f - 3000))
    print(f"  目的の方向の利得 |wᴴa| の 1 からのずれの最大 {np.max(np.abs(g_s[1:] - 1)):.1e}。"
          f"妨害の位置の利得 {20*np.log10(g_i[k1]):.1f} dB（1 kHz）、{20*np.log10(g_i[k3]):.1f} dB（3 kHz）")

    print("[5] MUSIC・NormMUSIC・SRP-PHAT（自由空間、音源2つ、マイク6本の円形アレイ）")
    n = FS * 2
    s = rng.standard_normal((2, n))
    doas = np.radians([60, 200])
    x = np.zeros((n, 6))
    for j in range(2):                                                # 周波数領域で分数の遅れを入れる
        S = np.fft.rfft(s[j])
        fr = np.fft.rfftfreq(n, 1 / FS)
        A = ar.steering_vector(mics, fr, azimuth=doas[j])
        x += np.fft.irfft(S[:, None] * A, n, axis=0)
    x += 0.05 * rng.standard_normal(x.shape)
    X = stft(x, L=nfft, S=nfft // 2)                                  # (F, T, M)
    Xp = np.transpose(X, (2, 0, 1))                                   # pra は (M, F, T)
    bins = np.arange(int(round(500 / FS * nfft)), int(round(4000 / FS * nfft)))
    fb = bins * FS / nfft
    for name, cls, kw, fn in [("MUSIC", pra.doa.MUSIC, {}, lambda: ar.music_spectrum(X[bins], mics, fb, grid, 2)),
                              ("NormMUSIC", pra.doa.MUSIC, {"frequency_normalization": True},
                               lambda: ar.music_spectrum(X[bins], mics, fb, grid, 2, normalize=True)),
                              ("SRP-PHAT", pra.doa.SRP, {}, lambda: ar.srp_phat_spectrum(X[bins], mics, fb, grid))]:
        d = cls(mics, FS, nfft, c=ar.C_SOUND, num_src=2, azimuth=grid, **kw)
        d.locate_sources(Xp, freq_bins=bins)
        ours = fn()
        if name == "SRP-PHAT":
            ours = ours / (6 * 5 / 2)                                 # pra は組の数 M(M−1)/2 で割っている
        rel = np.max(np.abs(ours - d.grid.values)) / np.max(np.abs(d.grid.values))
        found = np.sort(np.degrees(d.azimuth_recon))
        report(f"{name}（推定 {found[0]:.0f}°・{found[1]:.0f}°、真 60°・200°）相対", rel, 1e-9)

    print("[6] GCC-PHAT")
    n = FS
    s = rng.standard_normal(n + 100)
    for D in [3, -7, 25]:
        x1 = s[50:50 + n]
        x2 = s[50 - D:50 - D + n] + 0.1 * rng.standard_normal(n)      # x2 は x1 より D 標本遅れる
        tau, _, _ = ar.gcc_phat(x1, x2, FS)
        tau_pra = pra.experimental.tdoa(x1, x2, fs=FS, phat=True)     # pra は「x1 の x2 に対する遅れ」
        print(f"  D = {D:+3d} 標本：自前 {tau * FS:+.2f}、pra {tau_pra * FS:+.2f}（符号が逆の約束）")
        assert abs(tau * FS - D) < 1e-9 and abs(-tau_pra * FS - D) < 1e-9
    # 分数の遅れ 2.4 標本を、interp で細かく求める
    S = np.fft.rfft(s[:n])
    fr = np.fft.rfftfreq(n, 1 / FS)
    x2 = np.fft.irfft(S * np.exp(-2j * np.pi * fr * 2.4 / FS), n)
    for ip in [1, 4, 16]:
        tau, _, _ = ar.gcc_phat(s[:n], x2, FS, interp=ip)
        tau_pra = pra.experimental.tdoa(s[:n], x2, fs=FS, interp=ip, phat=True)
        print(f"  真の遅れ 2.4 標本、interp = {ip:2d}：自前 {tau * FS:.4f}、pra（符号を反転）{-tau_pra * FS:.4f}")
        assert abs(tau + tau_pra) < 1e-12
    print("すべて一致（許容誤差の範囲）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok("すべて一致" in cap.stdout and "★不一致" not in cap.stdout, "最後に「すべて一致」（[2]〜[5] がすべて OK）")
ok(abs(num(r"マイク1 に対する遅れ ([-\d.]+) µs") + 82.51) < 0.005, "[1] 遅れ −82.51 µs")
ok("0.8686+0.4955j" in cap.stdout and abs(num(r"位相差 ([-\d.]+) rad") - 0.5184) < 1e-4, "[1] a / a_1 = [1, 0.8686+0.4955j]、位相差 0.5184 rad")
diffs = [float(v) for v in re.findall(r"最大の差 ([\d.e+-]+)（許容", cap.stdout)]
ok(len(diffs) == 8 and max(diffs) < 1e-9, f"[2]〜[5] の最大の差がどれも 1e-9 より小さい（最大 {max(diffs):.1e}）")
ok("MUSIC（推定 60°・200°" in cap.stdout, "[5] MUSIC の推定が 60° と 200°")
g = re.findall(r"自前 ([-+\d.]+)、pra ([-+\d.]+)（符号が逆", cap.stdout)
ok(len(g) == 3 and all(float(a) == -float(b) for a, b in g), "[6] GCC-PHAT で、自前と pyroomacoustics の値が符号だけ逆")

### 手順 B の 3：わざと符号を間違えてみる（ノートブック版で足したセル）

教科書では `common/array.py` の `steering_vector` の `np.exp(-2j * …)` を `np.exp(+2j * …)` に変えて `check_array.py` を実行します。ここでは `array.py` を書きかえずに、符号を変えたベクトル（いまのベクトルの複素共役）を作って、[1] と [2] の照合だけをくり返します。

In [ ]:
# ノートブック版で足したセル：exp(−2j…) を exp(+2j…) にした版（＝複素共役）で [1]・[2] を見る
import pyroomacoustics as pra
sv_wrong = lambda *args, **kw: np.conj(ar.steering_vector(*args, **kw))
mics2 = ar.linear_array(2, 0.0566)
a = sv_wrong(mics2, [1000.0], azimuth=np.radians(60))[0]
ph_wrong = np.angle(a[1] / a[0])
a180 = ar.steering_vector(mics2, [1000.0], azimuth=np.radians(180 - 60))[0]   # 正しい式で、直線アレイの反対側 180° − θ
print(f"[1] 符号を変えた版の位相差 {ph_wrong:+.4f} rad（正しくは +0.5184）。正しい式で 120° の位相差 {np.angle(a180[1] / a180[0]):+.4f} rad")
mics = ar.circular_array(6, 0.05)
grid = np.linspace(0, 2 * np.pi, 72, endpoint=False)
mv = pra.doa.ModeVector(mics, 16000, 512, ar.C_SOUND, pra.doa.GridCircle(azimuth=grid))
k = np.arange(257)
d_wrong = np.max(np.abs(sv_wrong(mics, k * 16000 / 512, azimuth=grid) - np.transpose(mv[k, :, :], (0, 2, 1))))
print(f"[2] ModeVector との最大の差 {d_wrong:.2e}（許容 1e-09 → ★不一致。check_array.py ならここで AssertionError で止まる）")
ok(d_wrong > 1e-9, "手順 B の 3：[2] の ModeVector との照合で不一致になる")
ok(abs(ph_wrong + 0.5184) < 1e-4 and abs(np.angle(a180[1] / a180[0]) - ph_wrong) < 1e-9, "手順 B の 3：[1] の位相差の符号が逆になり、直線アレイでは 180° − θ の方向と同じになる")

## 2. 空間相関行列の固有値と音源の数（IV-5-3-3、手順 C、`scm_eig.py`）

数秒です。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-5：空間相関行列の固有値と音源の数

使い方: cd ~/home-lab/partIV/iv5 && uv run python scm_eig.py
4本の直線アレイ（5.66 cm）、1 kHz で、方向 70° と 120° の2つの音源（パワー 1 と 0.5）＋白色雑音（σ² = 0.01）。
  [1] 理論の空間相関行列 R = Σ_n σ_n² a_n a_nᴴ + σ² I の固有値（大きい2つが音源、残りは σ² に等しい）
  [2] T 個のスナップショット（フレーム）から推定した R̂ の固有値：雑音の固有値が σ² のまわりに広がる
  [3] 音源の数を MDL（Wax・Kailath 1985）と AIC で推定し、T ごとの正答率を出す（200 回）
  [4] 同じ2音源でも、方向が近いと2つ目の固有値が小さくなる（10° 差と 50° 差）
"""
import pathlib
import sys

import numpy as np

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import array as ar                                       # noqa: E402

M, D, F0 = 4, 0.0566, 1000.0
SIG2 = np.array([1.0, 0.5])
NOISE = 0.01


def true_R(degs):
    mics = ar.linear_array(M, D)
    A = ar.steering_vector(mics, [F0], azimuth=np.radians(degs))[0]   # (N, M)
    return (A.T * SIG2) @ A.conj() + NOISE * np.eye(M), A


def mdl_aic(lam, T):
    """固有値（大きい順）から音源の数を推定する。複素数のデータの Wax・Kailath の式"""
    Mm = len(lam)
    mdl, aic = [], []
    for k in range(Mm):
        tail = lam[k:]
        g = np.exp(np.mean(np.log(tail)))                            # 幾何平均
        a = np.mean(tail)                                            # 算術平均
        ll = -T * (Mm - k) * np.log(g / a)
        mdl.append(ll + 0.5 * k * (2 * Mm - k) * np.log(T))
        aic.append(2 * ll + 2 * k * (2 * Mm - k))
    return int(np.argmin(mdl)), int(np.argmin(aic))


def sample_R(R, T, rng):
    Lc = np.linalg.cholesky(R)
    z = (rng.standard_normal((M, T)) + 1j * rng.standard_normal((M, T))) / np.sqrt(2)
    x = Lc @ z                                                       # x ~ N_C(0, R)
    return x @ x.conj().T / T


def main():
    rng = np.random.default_rng(0)
    R, A = true_R([70, 120])
    lam = np.sort(np.linalg.eigvalsh(R))[::-1]
    print("[1] 理論の固有値（大きい順）：", np.round(lam, 4))
    print(f"    |a|² = M = {np.sum(np.abs(A[0])**2):.0f}。2つの方向のステアリングベクトルの内積の大きさ "
          f"|a₁ᴴa₂|/M = {abs(A[0].conj() @ A[1])/M:.3f}")
    print("[2] 推定した R̂ の固有値（1回の例）")
    for T in [10, 100, 1000]:
        l = np.sort(np.linalg.eigvalsh(sample_R(R, T, rng)))[::-1]
        print(f"    T = {T:4d}：", np.round(l, 4))
    print("[3] 音源の数の推定（真の数 2）の正答率（200 回）")
    for T in [5, 10, 30, 100, 1000]:
        ok_m = ok_a = 0
        for _ in range(200):
            l = np.sort(np.linalg.eigvalsh(sample_R(R, T, rng)))[::-1]
            km, ka = mdl_aic(l, T)
            ok_m += km == 2
            ok_a += ka == 2
        print(f"    T = {T:4d}：MDL {ok_m/2:5.1f}%、AIC {ok_a/2:5.1f}%")
    print("[4] 2つの音源の方向の差と、2番目の固有値")
    for degs in ([70, 120], [90, 100]):
        R2, A2 = true_R(degs)
        l = np.sort(np.linalg.eigvalsh(R2))[::-1]
        print(f"    {degs[0]}° と {degs[1]}°：固有値 {np.round(l, 4)}、|a₁ᴴa₂|/M = {abs(A2[0].conj() @ A2[1])/M:.3f}")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ev = [float(v) for v in re.findall(r"[-+\d.e]+", re.search(r"理論の固有値（大きい順）： \[([^\]]+)\]", cap.stdout).group(1))]
ok(len(ev) == 4 and all(abs(a - b) < 1e-4 for a, b in zip(ev, [4.9367, 1.0833, 0.01, 0.01])), f"[1] 理論の固有値が 4.9367・1.0833・0.01・0.01（{ev}。許容 1e-4）")
t10 = re.search(r"T =   10： \[([^\]]+)\]", cap.stdout).group(1).split()
ok(abs(float(t10[2]) - 0.01) > 3e-4 or abs(float(t10[3]) - 0.01) > 3e-4, f"[2] T = 10 で雑音の固有値が 0.01 からずれる（{t10[2]}・{t10[3]}。教科書 0.0104 と 0.0041）")
mdl = [float(v) for v in re.findall(r"MDL\s+([\d.]+)%", cap.stdout)]
aic = [float(v) for v in re.findall(r"AIC\s+([\d.]+)%", cap.stdout)]
ok(mdl[-1] >= 99 and mdl[0] < mdl[-1], f"[3] MDL の正答率が T とともに 100% に近づく {mdl}")
ok(80 <= aic[-1] <= 95, f"[3] AIC は 9 割前後で止まる {aic}")
ok(abs(num(r"90° と 100°：固有値 \[[\d.]+ ([\d.]+)") - 0.0637) < 1e-4, "[4] 90° と 100° の2つ目の固有値が 0.0637")

## 3. ビームパターンと空間エイリアシング（IV-5-3-4・3-7、手順 D、`beampattern.py`）

数十秒かかります。図の 1 枚目（`beampattern.png`：左から2本・正面、8本・正面、8本・軸の方向）で、正面に向けたとき周波数が高いほど主ローブが細いこと、軸の方向に向けた8本の 6000 Hz で 0° 以外にも 0 dB の山があることを見てください。2 枚目（`aliasing.png`）の右で、高い周波数ほど 0 dB の帯（黄色）が 0°・180° の側から増えてくることを見てください（手順 D の 6）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-5：遅延和ビームフォーマのビームパターン、主ローブの幅、グレーティングローブ、空間エイリアシング

使い方: cd ~/home-lab/partIV/iv5 && uv run python beampattern.py
  [1] 主ローブの幅（正面に向けたとき、最初のゼロの間）：数値と式 2·arcsin(λ/(Md)) の比較
  [2] 空間エイリアシングが始まる周波数 c/(2d) と、グレーティングローブが出る周波数
  [3] 2本のマイクの位相差 → 角度の答えがいくつあるか（1 kHz と 5 kHz）
  [4] 聞く：軸の方向（0°）から来るスイープを、正面（90°）に向けた 4本の遅延和に通す
出力: beampattern.png、aliasing.png、alias_sweep_mic1.wav、alias_sweep_ds.wav
"""
import pathlib
import sys

import numpy as np
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import plotting                                          # noqa: E402,F401  日本語フォント
from common.audio_io import save                                     # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000
C = ar.C_SOUND
D = 0.0566                    # 第7章の部屋のシミュレーションと同じ 5.66 cm


def first_null_width(M, d, f, steer=np.pi / 2):
    """数値で求めた主ローブの幅（最初のゼロからゼロまで）[度]"""
    mics = ar.linear_array(M, d)
    w = ar.ds_weights(ar.steering_vector(mics, [f], azimuth=steer))
    th = np.radians(np.linspace(0, 180, 180001))
    B = np.abs(ar.beam_pattern(w, mics, [f], th)[0])
    i0 = np.argmin(np.abs(th - steer))
    lo = i0
    while lo > 0 and B[lo - 1] < B[lo]:
        lo -= 1
    hi = i0
    while hi < len(B) - 1 and B[hi + 1] < B[hi]:
        hi += 1
    return np.degrees(th[hi] - th[lo]) if (lo > 0 and hi < len(B) - 1) else np.nan


def main():
    print("[1] 主ローブの幅（正面 90° に向けたとき、最初のゼロの間）")
    for M in [2, 4, 8]:
        for f in [500, 1000, 2000, 4000]:
            lam = C / f
            r = lam / (M * D)
            formula = 2 * np.degrees(np.arcsin(r)) if r <= 1 else np.nan
            num = first_null_width(M, D, f)
            s_f = "ゼロがない（主ローブが全体に広がる）" if np.isnan(formula) else f"式 {formula:6.1f}°"
            print(f"  M = {M}、d = {D*100:.2f} cm、{f:4d} Hz：数値 {num:6.1f}°、{s_f}")

    print("\n[2] 空間エイリアシング（間隔 d のとき、位相差が ±π を越え始める周波数 c/(2d)）")
    for d in [0.0283, 0.04, 0.0566, 0.1132]:
        print(f"  d = {d*100:5.2f} cm：c/(2d) = {C/(2*d):6.0f} Hz、正面に向けたときグレーティングローブが"
              f"軸の方向に出る周波数 c/d = {C/d:6.0f} Hz")

    print("\n[3] 2本（d = 5.66 cm）で、位相差から角度を求めると答えはいくつあるか")
    for f in [1000, 5000]:
        for th_true in [60]:
            dphi = 2 * np.pi * f * D * np.cos(np.radians(th_true)) / C    # マイク2 が進む位相
            dphi_obs = np.angle(np.exp(1j * dphi))                       # 観測できるのは −π〜π に折り返した値
            sols = []
            for k in range(-5, 6):
                c_ = (dphi_obs + 2 * np.pi * k) * C / (2 * np.pi * f * D)
                if -1 <= c_ <= 1:
                    sols.append(np.degrees(np.arccos(c_)))
            print(f"  {f} Hz、真の θ = {th_true}°：位相差 {dphi:.3f} rad（折り返すと {dphi_obs:.3f}）→ "
                  f"θ の候補 {', '.join(f'{s:.1f}°' for s in sorted(sols))}")

    # ---- 図：ビームパターン
    th = np.radians(np.linspace(0, 180, 721))
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
    for ax, (M, d, steer, ttl) in zip(axs, [(2, D, 90, "M = 2、d = 5.66 cm、正面へ"),
                                            (8, D, 90, "M = 8、d = 5.66 cm、正面へ"),
                                            (8, D, 0, "M = 8、d = 5.66 cm、軸の方向（0°）へ")]):
        mics = ar.linear_array(M, d)
        for f in [500, 1000, 2000, 4000, 6000]:
            w = ar.ds_weights(ar.steering_vector(mics, [f], azimuth=np.radians(steer)))
            B = np.abs(ar.beam_pattern(w, mics, [f], th)[0])
            ax.plot(np.degrees(th), 20 * np.log10(B + 1e-6), label=f"{f} Hz")
        ax.set(xlabel="到来方向 θ [度]", ylabel="利得 |B| [dB]", ylim=(-40, 3), title=ttl, xlim=(0, 180))
        ax.grid(True, alpha=0.3)
    axs[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("beampattern.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する

    # ---- 図：空間エイリアシング（位相差の折り返しと、周波数 × 方向の利得）
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
    f = np.linspace(0, FS / 2, 801)
    for th_deg in [0, 30, 60, 90]:
        ph = 2 * np.pi * f * D * np.cos(np.radians(th_deg)) / C
        axs[0].plot(f, np.angle(np.exp(1j * ph)), label=f"θ = {th_deg}°")
    axs[0].axvline(C / (2 * D), color="k", ls="--", lw=1)
    axs[0].text(C / (2 * D) + 80, -3.0, f"c/(2d) = {C/(2*D):.0f} Hz")
    axs[0].set(xlabel="周波数 [Hz]", ylabel="観測できる位相差 [rad]（−π〜π）",
               title="2本（5.66 cm）の位相差の折り返し")
    axs[0].legend(fontsize=8)
    mics = ar.linear_array(4, D)
    fk = np.linspace(50, FS / 2, 400)
    w = ar.ds_weights(ar.steering_vector(mics, fk, azimuth=np.pi / 2))
    B = 20 * np.log10(np.abs(ar.beam_pattern(w, mics, fk, th)) + 1e-6)
    im = axs[1].imshow(B, origin="lower", aspect="auto", extent=(0, 180, fk[0], fk[-1]),
                       vmin=-30, vmax=0, cmap="viridis")
    axs[1].set(xlabel="到来方向 θ [度]", ylabel="周波数 [Hz]", title="4本・5.66 cm・正面に向けた遅延和の利得 [dB]")
    fig.colorbar(im, ax=axs[1])
    for a in axs[:1]:
        a.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig("aliasing.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する

    # ---- 聞く：軸の方向（0°）から来るスイープ → 正面に向けた遅延和
    print("\n[4] 聞く：0° から来るスイープ（100 Hz → 7.9 kHz、6 秒）を、正面に向けた4本の遅延和に通す")
    n = 6 * FS
    t = np.arange(n) / FS
    f0, f1 = 100.0, 7900.0
    k = np.log(f1 / f0) / t[-1]
    sweep = np.sin(2 * np.pi * f0 * (np.exp(k * t) - 1) / k)          # 指数スイープ
    sweep *= np.minimum(1, np.minimum(t, t[-1] - t) / 0.05)          # 端を 50 ms でなめらかに
    S = np.fft.rfft(sweep, 2 * n)
    fr = np.fft.rfftfreq(2 * n, 1 / FS)
    A = ar.steering_vector(mics, fr, azimuth=0.0)                     # 0° から来る平面波
    x = np.fft.irfft(S[:, None] * A, 2 * n, axis=0)[:n]               # (n, 4) 各マイクの信号（分数の遅れ）
    Wf = ar.ds_weights(ar.steering_vector(mics, fr, azimuth=np.pi / 2))
    y = np.fft.irfft(np.einsum("fm,fm->f", Wf.conj(), np.fft.rfft(x, 2 * n, axis=0)), 2 * n)[:n]
    g = 0.03 / np.sqrt(np.mean(x[:, 0] ** 2))                        # マイク1 を RMS −30 dBFS にし、同じ倍率を出力にも
    save("alias_sweep_mic1.wav", x[:, 0] * g, FS)
    save("alias_sweep_ds.wav", y * g, FS)
    inst_f = f0 * np.exp(k * t)
    for fq in [1515, 3030, 4545, 6060, 7000]:
        i = np.argmin(np.abs(inst_f - fq))
        seg = slice(max(0, i - 200), i + 200)
        rms_ratio = np.sqrt(np.mean(y[seg] ** 2) / np.mean(x[seg, 0] ** 2))
        theo = np.abs(ar.beam_pattern(Wf[np.argmin(np.abs(fr - fq))][None], mics, [fq], [0.0])[0, 0])
        print(f"  {fq} Hz 付近（{t[i]:.2f} 秒）：出力/入力の RMS 比 {20*np.log10(rms_ratio+1e-9):6.1f} dB"
              f"（理論 {20*np.log10(theo+1e-9):6.1f} dB）")
    print("beampattern.png、aliasing.png、alias_sweep_mic1.wav、alias_sweep_ds.wav を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
w = re.findall(r"数値\s+([\d.]+)°、式\s+([\d.]+)°", cap.stdout)
ok(len(w) >= 6 and all(a == b for a, b in w), "[1] 数値で探した主ローブの幅と式が小数1桁まで一致")
ok(num(r"M = 8、d = 5\.66 cm、2000 Hz：数値\s+([\d.]+)°") == 44.5, "[1] 8本・2 kHz で 44.5°")
ok(num(r"d =  5\.66 cm：c/\(2d\) =\s+(\d+) Hz") == 3030, "[2] 5.66 cm の c/(2d) が 3030 Hz")
ok("5000 Hz、真の θ = 60°" in cap.stdout and "θ の候補 60.0°, 135.4°" in cap.stdout, "[3] 5 kHz の候補が 60.0° と 135.4° の2つ")
r = {f: num(rf"{f} Hz 付近.*?RMS 比\s+([-\d.]+) dB") for f in (1515, 3030, 4545, 6060)}
ok(abs(r[6060]) < 0.5, "[4] 6060 Hz 付近の「出力/入力」が 0 dB（グレーティングローブ）")
ok(all(-47 < r[f] < -33 for f in (1515, 3030, 4545)), f"[4] 1515・3030・4545 Hz 付近で −35〜−45 dB 前後 {r}")

1本で録ったスイープと、遅延和の出力を聞き比べます（手順 D の 7）。遅延和の出力では、スイープが上がるにつれて小さくなり（途中で何度か消え）、終わり近く（約 5.6 秒、6 kHz 付近）で急に元の大きさに戻ります。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['alias_sweep_mic1.wav', 'alias_sweep_ds.wav']:
    print(f)
    listen_file(f)

## 4. 遅延和と MVDR を比べる（IV-5-3-5、手順 E、`beamform_demo.py`）

数秒です（初回は libri1・libri2 をダウンロードします）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['beamform_demo.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-5：遅延和（DS）と MVDR を部屋のシミュレーションで比べ、耳で聞く

使い方: cd ~/home-lab/partIV/iv5 && uv run python beamform_demo.py [--rt60 0.2]
部屋（6×5×3 m）に、目的の話し声（100°）と妨害の話し声（40°）を置き、4本の直線アレイ（5.66 cm 間隔）で録る。
各マイクに独立な雑音（SNR 20 dB）も足す。STFT は L = 1024、S = 256。
比べる重み（どれも周波数ごと。同じ重みを、目的・妨害・雑音の成分に別々に掛けて評価する）
  DS            : 幾何（平面波）のステアリングベクトル a(100°) / M
  MVDR(白)      : R_n = σ²I（マイクごとに独立な雑音だけを知っている）→ DS と同じになるはず
  MVDR(正解)    : R_n = 妨害＋雑音の空間相関行列（正解の成分から作る。oracle）
  MPDR          : R = 混合音の空間相関行列（目的の音も入っている）。ステアリングベクトルが少しでもずれると、
                  目的の音を「雑音」とみなして消してしまう（信号の打ち消し）
  MPDR＋対角装荷 : R + εI（ε = loading × tr(R)/M）。loading を変えて DS と MPDR の間を動く
  MVDR(RTF)     : 目的の像の空間相関行列の最大の固有ベクトル（マイク1 を 1 にそろえた相対伝達関数）と R_n(正解)
評価（100 Hz〜7 kHz の帯域、時間領域）：SIR の改善、SNR の改善、SDR（基準は目的の音のマイク1 の像。fast_bss_eval）、
       白色雑音利得 WNG の帯域の中央値
出力: bf_*.wav（聞き比べ。すべて同じ倍率で保存するので、大きさも比べられる）、beamform.png
"""
import argparse
import pathlib
import sys

import numpy as np
import pyroomacoustics as pra
import fast_bss_eval
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import plotting                                          # noqa: E402,F401
from common.stft import stft, istft                                  # noqa: E402
from common.audio_io import save                                     # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000
L, S = 1024, 256
M, D = 4, 0.0566
TGT_DEG, ITF_DEG = 100, 40
SEC = 8.0


def load_speech(n):
    import librosa
    out = []
    for name in ("libri1", "libri2"):
        s, _ = librosa.load(librosa.ex(name), sr=FS, mono=True)
        s = s[:n]
        out.append(s / np.std(s))
    return out


def simulate(rt60, rng):
    n = int(SEC * FS)
    s_t, s_i = load_speech(n)
    dim = np.array([6.0, 5.0, 3.0])
    e_abs, max_order = pra.inverse_sabine(rt60, dim)
    room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    center = np.array([3.1, 2.4, 1.4])
    xy = ar.linear_array(M, D)                                       # 中心が原点、x 軸に沿う (2, M)
    room.add_microphone_array(np.vstack([xy + center[:2, None], np.full(M, center[2])]))
    for deg, s in ((TGT_DEG, s_t), (ITF_DEG, s_i)):
        u = ar.unit_vector(np.radians(deg), dim=3)
        room.add_source(center + 1.5 * u, signal=s)
    pre = room.simulate(return_premix=True)                          # (2, M, 時間)
    tgt, itf = pre[0, :, :n].T, pre[1, :, :n].T                       # (n, M)
    sig = np.mean(tgt[:, 0] ** 2)
    noise = rng.standard_normal((n, M)) * np.sqrt(sig / 10 ** (20 / 10))   # マイクごとに独立、SNR 20 dB
    return xy, tgt, itf, noise


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rt60", type=float, default=0.2)
    a = ap.parse_args()
    rng = np.random.default_rng(0)
    xy, tgt, itf, noise = simulate(a.rt60, rng)
    n = tgt.shape[0]
    Xt, Xi, Xn = stft(tgt, L=L, S=S), stft(itf, L=L, S=S), stft(noise, L=L, S=S)
    Xm = Xt + Xi + Xn
    f = np.arange(L // 2 + 1) * FS / L
    band = (f >= 100) & (f <= 7000)

    a_geo = ar.steering_vector(xy, f, azimuth=np.radians(TGT_DEG))    # 平面波のステアリングベクトル (F, M)
    Rn_true = ar.scm(Xi + Xn)
    sigma2 = np.real(np.trace(ar.scm(Xn), axis1=1, axis2=2)) / M
    Rt = ar.scm(Xt)
    a_rtf = np.linalg.eigh(Rt)[1][:, :, -1]
    a_rtf = a_rtf / a_rtf[:, :1]                                     # マイク1 の成分を 1 にそろえる
    weights = {
        "DS": ar.ds_weights(a_geo),
        "MVDR(白)": ar.mvdr_weights(a_geo, sigma2[:, None, None] * np.eye(M)[None]),
        "MVDR(正解)": ar.mvdr_weights(a_geo, Rn_true, loading=1e-6),
        "MPDR": ar.mvdr_weights(a_geo, ar.scm(Xm), loading=1e-6),
    }
    for ld in [1e-3, 1e-2, 1e-1, 1.0]:
        weights[f"MPDR+装荷{ld:g}"] = ar.mvdr_weights(a_geo, ar.scm(Xm), loading=ld)
    weights["MVDR(RTF)"] = ar.mvdr_weights(a_rtf, Rn_true, loading=1e-6)
    print(f"RT60（設定）{a.rt60} s、マイク {M} 本・{D*100:.2f} cm、目的 {TGT_DEG}°、妨害 {ITF_DEG}°、雑音 SNR 20 dB")
    print(f"MVDR(白) と DS の重みの最大の差：{np.max(np.abs(weights['MVDR(白)'] - weights['DS'])):.1e}"
          "（R_n = σ²I なら MVDR = DS）")

    def td(Y):
        return istft(Y, L, S, length=n)

    ref = tgt[:, 0]
    def bandpow(x):
        Xb = stft(x, L=L, S=S)
        return np.sum(np.abs(Xb[band]) ** 2)
    sir_in = 10 * np.log10(bandpow(tgt[:, 0]) / bandpow(itf[:, 0]))
    snr_in = 10 * np.log10(bandpow(tgt[:, 0]) / bandpow(noise[:, 0]))
    sdr_in = float(fast_bss_eval.sdr(ref[None], (tgt[:, 0] + itf[:, 0] + noise[:, 0])[None])[0])
    print(f"\n入力（マイク1）：SIR {sir_in:.1f} dB、SNR {snr_in:.1f} dB、SDR {sdr_in:.1f} dB")
    print(f"{'方法':14s} {'SIR改善':>7s} {'SNR改善':>7s} {'SDR':>6s} {'目的の音の大きさ':>10s} {'WNG中央値':>9s}")
    outs = {}
    for name, W in weights.items():
        yt, yi, yn = td(ar.apply_weights(W, Xt)), td(ar.apply_weights(W, Xi)), td(ar.apply_weights(W, Xn))
        y = yt + yi + yn
        outs[name] = y
        sir = 10 * np.log10(bandpow(yt) / bandpow(yi)) - sir_in
        snr = 10 * np.log10(bandpow(yt) / bandpow(yn)) - snr_in
        sdr = float(fast_bss_eval.sdr(ref[None], y[None])[0])
        gain_t = 10 * np.log10(bandpow(yt) / bandpow(tgt[:, 0]))
        wng = 10 * np.log10(np.median(ar.white_noise_gain(W[band], a_geo[band] if "RTF" not in name else a_rtf[band])))
        print(f"{name:14s} {sir:7.1f} {snr:7.1f} {sdr:6.1f} {gain_t:10.1f} dB {wng:9.1f} dB")

    # 聞き比べ用：混合音（マイク1）を RMS −30 dBFS にする倍率を、すべてに同じく掛ける
    mix1 = tgt[:, 0] + itf[:, 0] + noise[:, 0]
    files = {"bf_0_mix_mic1.wav": mix1, "bf_1_target_ref.wav": ref, "bf_2_ds.wav": outs["DS"],
             "bf_3_mvdr.wav": outs["MVDR(正解)"], "bf_4_mpdr.wav": outs["MPDR"],
             "bf_5_mpdr_dl0.1.wav": outs["MPDR+装荷0.1"], "bf_6_mvdr_rtf.wav": outs["MVDR(RTF)"]}
    g = 0.03 / np.sqrt(np.mean(mix1 ** 2))
    peak = max(np.max(np.abs(x)) for x in files.values()) * g
    if peak > 0.99:
        g *= 0.99 / peak
    for fn, x in files.items():
        save(fn, x * g, FS)
    print("\n" + "、".join(files) + " を保存しました（同じ倍率。まず Mac の音量を最小にしてから聞く）")

    # 図：1 kHz と 3 kHz のビームパターン（DS・MVDR(正解)・MPDR）
    th = np.radians(np.linspace(0, 180, 361))
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, fq in zip(axs, [1000, 3000]):
        k = int(round(fq * L / FS))
        for name in ["DS", "MVDR(正解)", "MPDR", "MPDR+装荷0.1"]:
            B = ar.beam_pattern(weights[name][k][None], xy, [f[k]], th)[0]
            ax.plot(np.degrees(th), 20 * np.log10(np.abs(B) + 1e-6), label=name)
        ax.axvline(TGT_DEG, color="k", ls="--", lw=1)
        ax.axvline(ITF_DEG, color="r", ls=":", lw=1)
        ax.set(xlabel="方向 θ [度]", ylabel="利得 [dB]", ylim=(-40, 15), title=f"{f[k]:.0f} Hz（破線：目的、点線：妨害）")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("beamform.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("beamform.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"MVDR\(白\) と DS の重みの最大の差：([\d.e+-]+)") < 1e-15, "MVDR(白) と DS の重みの最大の差が 1e-15 より小さい")
row = lambda name: [float(v) for v in re.search(rf"^{re.escape(name)}\s+([-\d.]+)\s+([-\d.]+)\s+([-\d.]+)\s+([-\d.]+) dB", cap.stdout, re.M).groups()]
ds, mv, mp = row("DS"), row("MVDR(正解)"), row("MPDR")
ok(abs(ds[0] - 1) < 0.5 and abs(mv[0] - 13) < 1, f"SIR の改善が DS で約 1 dB（{ds[0]}）、MVDR(正解) で約 13 dB（{mv[0]}）")
ok(abs(mp[3] + 2) < 0.7 and abs(mp[1] + 11) < 1.5, f"MPDR の目的の音の大きさが約 −2 dB（{mp[3]}）、SNR の改善が約 −11 dB（{mp[1]}）")
dl = [row(f"MPDR+装荷{x}") for x in ("0.001", "0.01", "0.1", "1")]
ok(abs(dl[-1][0] - ds[0]) < abs(mp[0] - ds[0]) and abs(dl[-1][3] - ds[3]) < abs(mp[3] - ds[3]), "対角装荷を大きくすると、MPDR の値が DS に近づく")

聞き比べます（手順 E の 9）：混合音 → DS → MVDR → MPDR → MPDR＋対角装荷 0.1 → 正解。DS は混合音とほとんど変わらず、MVDR では妨害の声がはっきり小さくなり、サーッという雑音が少し増えます。MPDR は目的の声が細く・小さくなります。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['bf_0_mix_mic1.wav', 'bf_2_ds.wav', 'bf_3_mvdr.wav', 'bf_4_mpdr.wav', 'bf_5_mpdr_dl0.1.wav', 'bf_1_target_ref.wav']:
    print(f)
    listen_file(f)

### 条件を変える（手順 E の 10）

`--rt60 0.5` で実行し、表を上の RT60 0.2 s と比べます。MVDR(正解) と MVDR(RTF) の差、MPDR の信号の打ち消しがどう変わるかをノートに書いてください（wav は上書きされます）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['beamform_demo.py', '--rt60', '0.5']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-5：遅延和（DS）と MVDR を部屋のシミュレーションで比べ、耳で聞く

使い方: cd ~/home-lab/partIV/iv5 && uv run python beamform_demo.py [--rt60 0.2]
部屋（6×5×3 m）に、目的の話し声（100°）と妨害の話し声（40°）を置き、4本の直線アレイ（5.66 cm 間隔）で録る。
各マイクに独立な雑音（SNR 20 dB）も足す。STFT は L = 1024、S = 256。
比べる重み（どれも周波数ごと。同じ重みを、目的・妨害・雑音の成分に別々に掛けて評価する）
  DS            : 幾何（平面波）のステアリングベクトル a(100°) / M
  MVDR(白)      : R_n = σ²I（マイクごとに独立な雑音だけを知っている）→ DS と同じになるはず
  MVDR(正解)    : R_n = 妨害＋雑音の空間相関行列（正解の成分から作る。oracle）
  MPDR          : R = 混合音の空間相関行列（目的の音も入っている）。ステアリングベクトルが少しでもずれると、
                  目的の音を「雑音」とみなして消してしまう（信号の打ち消し）
  MPDR＋対角装荷 : R + εI（ε = loading × tr(R)/M）。loading を変えて DS と MPDR の間を動く
  MVDR(RTF)     : 目的の像の空間相関行列の最大の固有ベクトル（マイク1 を 1 にそろえた相対伝達関数）と R_n(正解)
評価（100 Hz〜7 kHz の帯域、時間領域）：SIR の改善、SNR の改善、SDR（基準は目的の音のマイク1 の像。fast_bss_eval）、
       白色雑音利得 WNG の帯域の中央値
出力: bf_*.wav（聞き比べ。すべて同じ倍率で保存するので、大きさも比べられる）、beamform.png
"""
import argparse
import pathlib
import sys

import numpy as np
import pyroomacoustics as pra
import fast_bss_eval
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import plotting                                          # noqa: E402,F401
from common.stft import stft, istft                                  # noqa: E402
from common.audio_io import save                                     # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000
L, S = 1024, 256
M, D = 4, 0.0566
TGT_DEG, ITF_DEG = 100, 40
SEC = 8.0


def load_speech(n):
    import librosa
    out = []
    for name in ("libri1", "libri2"):
        s, _ = librosa.load(librosa.ex(name), sr=FS, mono=True)
        s = s[:n]
        out.append(s / np.std(s))
    return out


def simulate(rt60, rng):
    n = int(SEC * FS)
    s_t, s_i = load_speech(n)
    dim = np.array([6.0, 5.0, 3.0])
    e_abs, max_order = pra.inverse_sabine(rt60, dim)
    room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    center = np.array([3.1, 2.4, 1.4])
    xy = ar.linear_array(M, D)                                       # 中心が原点、x 軸に沿う (2, M)
    room.add_microphone_array(np.vstack([xy + center[:2, None], np.full(M, center[2])]))
    for deg, s in ((TGT_DEG, s_t), (ITF_DEG, s_i)):
        u = ar.unit_vector(np.radians(deg), dim=3)
        room.add_source(center + 1.5 * u, signal=s)
    pre = room.simulate(return_premix=True)                          # (2, M, 時間)
    tgt, itf = pre[0, :, :n].T, pre[1, :, :n].T                       # (n, M)
    sig = np.mean(tgt[:, 0] ** 2)
    noise = rng.standard_normal((n, M)) * np.sqrt(sig / 10 ** (20 / 10))   # マイクごとに独立、SNR 20 dB
    return xy, tgt, itf, noise


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rt60", type=float, default=0.2)
    a = ap.parse_args()
    rng = np.random.default_rng(0)
    xy, tgt, itf, noise = simulate(a.rt60, rng)
    n = tgt.shape[0]
    Xt, Xi, Xn = stft(tgt, L=L, S=S), stft(itf, L=L, S=S), stft(noise, L=L, S=S)
    Xm = Xt + Xi + Xn
    f = np.arange(L // 2 + 1) * FS / L
    band = (f >= 100) & (f <= 7000)

    a_geo = ar.steering_vector(xy, f, azimuth=np.radians(TGT_DEG))    # 平面波のステアリングベクトル (F, M)
    Rn_true = ar.scm(Xi + Xn)
    sigma2 = np.real(np.trace(ar.scm(Xn), axis1=1, axis2=2)) / M
    Rt = ar.scm(Xt)
    a_rtf = np.linalg.eigh(Rt)[1][:, :, -1]
    a_rtf = a_rtf / a_rtf[:, :1]                                     # マイク1 の成分を 1 にそろえる
    weights = {
        "DS": ar.ds_weights(a_geo),
        "MVDR(白)": ar.mvdr_weights(a_geo, sigma2[:, None, None] * np.eye(M)[None]),
        "MVDR(正解)": ar.mvdr_weights(a_geo, Rn_true, loading=1e-6),
        "MPDR": ar.mvdr_weights(a_geo, ar.scm(Xm), loading=1e-6),
    }
    for ld in [1e-3, 1e-2, 1e-1, 1.0]:
        weights[f"MPDR+装荷{ld:g}"] = ar.mvdr_weights(a_geo, ar.scm(Xm), loading=ld)
    weights["MVDR(RTF)"] = ar.mvdr_weights(a_rtf, Rn_true, loading=1e-6)
    print(f"RT60（設定）{a.rt60} s、マイク {M} 本・{D*100:.2f} cm、目的 {TGT_DEG}°、妨害 {ITF_DEG}°、雑音 SNR 20 dB")
    print(f"MVDR(白) と DS の重みの最大の差：{np.max(np.abs(weights['MVDR(白)'] - weights['DS'])):.1e}"
          "（R_n = σ²I なら MVDR = DS）")

    def td(Y):
        return istft(Y, L, S, length=n)

    ref = tgt[:, 0]
    def bandpow(x):
        Xb = stft(x, L=L, S=S)
        return np.sum(np.abs(Xb[band]) ** 2)
    sir_in = 10 * np.log10(bandpow(tgt[:, 0]) / bandpow(itf[:, 0]))
    snr_in = 10 * np.log10(bandpow(tgt[:, 0]) / bandpow(noise[:, 0]))
    sdr_in = float(fast_bss_eval.sdr(ref[None], (tgt[:, 0] + itf[:, 0] + noise[:, 0])[None])[0])
    print(f"\n入力（マイク1）：SIR {sir_in:.1f} dB、SNR {snr_in:.1f} dB、SDR {sdr_in:.1f} dB")
    print(f"{'方法':14s} {'SIR改善':>7s} {'SNR改善':>7s} {'SDR':>6s} {'目的の音の大きさ':>10s} {'WNG中央値':>9s}")
    outs = {}
    for name, W in weights.items():
        yt, yi, yn = td(ar.apply_weights(W, Xt)), td(ar.apply_weights(W, Xi)), td(ar.apply_weights(W, Xn))
        y = yt + yi + yn
        outs[name] = y
        sir = 10 * np.log10(bandpow(yt) / bandpow(yi)) - sir_in
        snr = 10 * np.log10(bandpow(yt) / bandpow(yn)) - snr_in
        sdr = float(fast_bss_eval.sdr(ref[None], y[None])[0])
        gain_t = 10 * np.log10(bandpow(yt) / bandpow(tgt[:, 0]))
        wng = 10 * np.log10(np.median(ar.white_noise_gain(W[band], a_geo[band] if "RTF" not in name else a_rtf[band])))
        print(f"{name:14s} {sir:7.1f} {snr:7.1f} {sdr:6.1f} {gain_t:10.1f} dB {wng:9.1f} dB")

    # 聞き比べ用：混合音（マイク1）を RMS −30 dBFS にする倍率を、すべてに同じく掛ける
    mix1 = tgt[:, 0] + itf[:, 0] + noise[:, 0]
    files = {"bf_0_mix_mic1.wav": mix1, "bf_1_target_ref.wav": ref, "bf_2_ds.wav": outs["DS"],
             "bf_3_mvdr.wav": outs["MVDR(正解)"], "bf_4_mpdr.wav": outs["MPDR"],
             "bf_5_mpdr_dl0.1.wav": outs["MPDR+装荷0.1"], "bf_6_mvdr_rtf.wav": outs["MVDR(RTF)"]}
    g = 0.03 / np.sqrt(np.mean(mix1 ** 2))
    peak = max(np.max(np.abs(x)) for x in files.values()) * g
    if peak > 0.99:
        g *= 0.99 / peak
    for fn, x in files.items():
        save(fn, x * g, FS)
    print("\n" + "、".join(files) + " を保存しました（同じ倍率。まず Mac の音量を最小にしてから聞く）")

    # 図：1 kHz と 3 kHz のビームパターン（DS・MVDR(正解)・MPDR）
    th = np.radians(np.linspace(0, 180, 361))
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, fq in zip(axs, [1000, 3000]):
        k = int(round(fq * L / FS))
        for name in ["DS", "MVDR(正解)", "MPDR", "MPDR+装荷0.1"]:
            B = ar.beam_pattern(weights[name][k][None], xy, [f[k]], th)[0]
            ax.plot(np.degrees(th), 20 * np.log10(np.abs(B) + 1e-6), label=name)
        ax.axvline(TGT_DEG, color="k", ls="--", lw=1)
        ax.axvline(ITF_DEG, color="r", ls=":", lw=1)
        ax.set(xlabel="方向 θ [度]", ylabel="利得 [dB]", ylim=(-40, 15), title=f"{f[k]:.0f} Hz（破線：目的、点線：妨害）")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("beamform.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("beamform.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
print(cap.stdout.split("bf_0_mix")[0])   # 表だけを表示する（図は省く）
ok("RT60（設定）0.5 s" in cap.stdout and "MVDR(RTF)" in cap.stdout, "--rt60 0.5 で表が出た（数値の比べ方は自分でノートに）")

## 5. 窓の長さと瞬時混合近似（IV-5-3-8、手順 F の 11、`narrowband.py`）

十数秒かかります。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-5：周波数領域の瞬時混合近似 x_ft ≈ a_f s_ft は、どんな窓の長さで成り立つか

使い方: cd ~/home-lab/partIV/iv5 && uv run python narrowband.py
部屋（6×5×3 m、残響時間 RT60 を変える）に音源を1つ置き、2本（5.66 cm）で録った音（音源の像）を作る。
窓の長さ L を変えて、次の3つを測る。
  [1] 近似の誤差 E(L) = Σ|x_ft − ĥ_f s_ft|² / Σ|x_ft|²（ĥ_f は周波数ごとの最小二乗の当てはめ）[dB]
  [2] 1つの音源の空間相関行列が「階数1」からどれだけ離れるか：固有値の比 λ₂/λ₁ の中央値 [dB]
  [3] 6 秒の音でのフレームの数 T（窓を長くすると、推定に使える数が減る）
あわせて、インパルス応答のうち「直接音から L 標本より後ろ」にあるエネルギーの割合も出す。
出力: narrowband.png、narrowband.csv
"""
import pathlib
import sys

import numpy as np
import pandas as pd
import pyroomacoustics as pra
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import plotting                                          # noqa: E402,F401
from common.stft import stft                                         # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000
DIM = np.array([6.0, 5.0, 3.0])
RT60S = [0.0, 0.15, 0.3, 0.5, 0.8]
LS = [256, 512, 1024, 2048, 4096, 8192]
SEC = 6.0


def source_signal(n, rng):
    """音源：librosa の libri1（話し声）。使えなければ白色雑音"""
    try:
        import librosa
        s, _ = librosa.load(librosa.ex("libri1"), sr=FS, mono=True)
        s = s[:n]
    except Exception as e:
        print(f"libri1 が使えないので白色雑音にします（{type(e).__name__}）")
        s = rng.standard_normal(n)
    return s / np.std(s)


def simulate(s, rt60):
    """音源を1つ置いた部屋で、2本のマイク（5.66 cm）の信号（音源の像）とインパルス応答を返す"""
    if rt60 == 0:
        room = pra.ShoeBox(DIM, fs=FS, max_order=0)                   # 反射なし（自由空間と同じ）
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, DIM)
        room = pra.ShoeBox(DIM, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    center = np.array([3.1, 2.4, 1.4])
    mics = np.c_[center - [0.0283, 0, 0], center + [0.0283, 0, 0]]
    room.add_microphone_array(mics)
    room.add_source(center + 1.5 * np.array([np.cos(np.radians(50)), np.sin(np.radians(50)), 0.1]), signal=s)
    room.compute_rir()
    room.simulate()
    x = room.mic_array.signals[:, : len(s)].T                         # (n, 2)
    h = room.rir[0][0]
    return x, h


def tail_fraction(h, L):
    """インパルス応答のエネルギーのうち、直接音（最大の点）から L 標本より後ろにある割合"""
    i0 = int(np.argmax(np.abs(h)))
    e = h ** 2
    return float(e[i0 + L:].sum() / e.sum())


def main():
    rng = np.random.default_rng(0)
    n = int(SEC * FS)
    s = source_signal(n, rng)
    rows = []
    for rt60 in RT60S:
        x, h = simulate(s, rt60)
        rt_meas = pra.experimental.measure_rt60(h, fs=FS) if rt60 > 0 else 0.0
        i0 = int(np.argmax(np.abs(h)))                                # 直接音の位置（約 110 標本）
        for L in LS:
            S = L // 4
            X = stft(x, L=L, S=S)                                     # (F, T, 2)
            # [1] 周波数ごとの最小二乗 ĥ_f = Σ_t x s* / Σ_t |s|²（マイク1）。
            #     音源の時刻の基準を「そのまま」と「直接音にそろえる（i0 標本遅らせる）」の2通り
            err = {}
            for key, s_ref in (("raw", s), ("aligned", np.r_[np.zeros(i0), s[: n - i0]])):
                Xs = stft(s_ref, L=L, S=S)                            # (F, T)
                hf = np.sum(X[:, :, 0] * Xs.conj(), axis=1) / (np.sum(np.abs(Xs) ** 2, axis=1) + 1e-12)
                err[key] = np.sum(np.abs(X[:, :, 0] - hf[:, None] * Xs) ** 2) / np.sum(np.abs(X[:, :, 0]) ** 2)
            # [2] 1つの音源の空間相関行列の固有値の比（100 Hz〜4 kHz の中央値）
            R = ar.scm(X)
            lam = np.linalg.eigvalsh(R)                               # 小さい順 (F, 2)
            f = np.arange(L // 2 + 1) * FS / L
            band = (f >= 100) & (f <= 4000)
            ratio = np.median(lam[band, 0] / lam[band, 1])
            rows.append(dict(rt60=rt60, rt60_meas=rt_meas, L=L, ms=1000 * L / FS, T=X.shape[1],
                             err_db=10 * np.log10(err["aligned"]),
                             err_raw_db=10 * np.log10(err["raw"]), i0=i0, eig_ratio_db=10 * np.log10(ratio),
                             tail_db=10 * np.log10(tail_fraction(h, L) + 1e-12)))
    d = pd.DataFrame(rows)
    d.to_csv("narrowband.csv", index=False)
    pd.set_option("display.width", 140)
    print("[1] 近似の誤差 E(L) [dB]（行：設定した RT60 [s]、列：窓の長さ L [標本]（16 kHz））")
    print("    音源の時刻を直接音にそろえた場合（本文の表）")
    print(d.pivot(index="rt60", columns="L", values="err_db").round(1).to_string())
    print(f"    そろえない場合（直接音までの遅れ {d['i0'].iloc[0]} 標本もそのまま近似に入る）")
    print(d.pivot(index="rt60", columns="L", values="err_raw_db").round(1).to_string())
    print("\n    参考：インパルス応答のうち、直接音から L 標本より後ろのエネルギーの割合 [dB]")
    print(d.pivot(index="rt60", columns="L", values="tail_db").round(1).to_string())
    print("\n[2] 1つの音源の空間相関行列の固有値の比 λ₂/λ₁ の中央値 [dB]（−∞ に近いほど階数1）")
    print(d.pivot(index="rt60", columns="L", values="eig_ratio_db").round(1).to_string())
    print("\n[3] 6 秒の音のフレームの数 T（S = L/4）：",
          "、".join(f"L = {L}（{1000*L/FS:.0f} ms）→ {t}" for L, t in d.groupby("L")["T"].first().items()))
    print("\n    シュレーダーの方法で測った RT60 [s]：",
          "、".join(f"設定 {r} → {m:.2f}" for r, m in d.groupby("rt60")["rt60_meas"].first().items() if r > 0))

    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
    for rt60, g in d.groupby("rt60"):
        axs[0].semilogx(g["ms"], g["err_db"], "o-", label=f"RT60 {rt60:.2f} s")
        axs[1].semilogx(g["ms"], g["eig_ratio_db"], "o-", label=f"RT60 {rt60:.2f} s")
    axs[0].set(xlabel="窓の長さ [ms]", ylabel="近似の誤差 E [dB]", title="x_ft ≈ h_f s_ft の誤差")
    axs[1].set(xlabel="窓の長さ [ms]", ylabel="λ₂/λ₁ の中央値 [dB]", title="1つの音源の空間相関行列の「階数1らしさ」")
    for a in axs:
        a.grid(True, which="both", alpha=0.3)
        a.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("narrowband.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("narrowband.png、narrowband.csv を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
import pandas as pd
nb = pd.read_csv("narrowband.csv")
print(nb.head())
tabs = cap.stdout.split("そろえない場合")
def table(txt):
    t = {}
    for m in re.finditer(r"^(\d\.\d\d)((?:\s+-?[\d.]+){6})$", txt, re.M):
        t[float(m.group(1))] = [float(v) for v in m.group(2).split()]
    return t
al, na = table(tabs[0].split("[1]")[1]), table(tabs[1].split("参考")[0])
ok(all(all(a > b for a, b in zip(r, r[1:])) for r in al.values()), "[1] 直接音にそろえた場合：どの行も右（長い窓）ほど誤差が小さい")
cols = list(zip(*[al[k] for k in sorted(al)]))
ok(all(all(a < b for a, b in zip(c, c[1:])) for c in cols), "[1] 下（長い残響）ほど誤差が大きい")
ok(abs(al[0.30][3] + 9) < 1, f"[1] RT60 0.3 s・2048 点で約 −9 dB（{al[0.30][3]}）")
ok(abs(na[0.00][0] + 1) < 0.5, f"[1] そろえない場合は、反射なしでも 256 点で約 −1 dB（{na[0.00][0]}）")
eg = table(cap.stdout.split("[2]")[1].split("[3]")[0])
ok(all(all(a > b for a, b in zip(r, r[1:])) for r in eg.values()), "[2] 固有値の比も同じ傾向（右ほど小さい）")
T = [int(v) for v in re.findall(r"ms）→ (\d+)", cap.stdout.split("[3]")[1])]
ok(all(abs(T[i] / T[i + 1] - 2) < 0.01 for i in range(len(T) - 1)), f"[3] フレームの数が窓の長さに反比例して減る {T}")

## 6. 非同期録音のずれ（IV-5-3-10、手順 H、`sfo.py`）

1 分ほどかかります。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-5：非同期録音の標本化周波数のずれ（SFO）が、ステアリングベクトルと空間相関行列にどう効くか

使い方: cd ~/home-lab/partIV/iv5 && uv run python sfo.py
部屋（6×5×3 m、RT60 0.2 s）に話し声を2つ（目的・妨害）置き、0.6 m 離れた2台の録音機（各1本）で録る。
2台目だけ標本化周波数が fs(1+ε) だったとして、次を調べる（ε は ppm = 10⁻⁶）。
  [1] ずれの大きさの計算（式の数値例）
  [2] GCC-PHAT を 1 秒ごとに求めると、時間差が時間に比例して増える。傾きを回帰で推定する
  [3] 空間相関行列の非対角成分のコヒーレンス |R12|/√(R11 R22) と、固有値の比が、
      平均するフレームの数 T とともにどう落ちるか（理論：ディリクレ核 |sin(Tα/2)/(T sin(α/2))|）
  [4] 最初の状態で作った MVDR を使い続けると、妨害の抑え方（SIR の改善）が時間とともに崩れる。
      正しい ε で STFT の位相を戻すと（sfo_phase）回復する
  [5] ε を知らなくても、尤度（Σ_f log det R̂_f の最小）で ε を推定できる（Miyabe ら 2015 の考え方を単純にしたもの）
出力: sfo.png
"""
import pathlib
import sys

import numpy as np
import pyroomacoustics as pra
import statsmodels.api as sm
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import plotting                                          # noqa: E402,F401
from common.stft import stft                                         # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000
L, S = 1024, 256
SEC = 14.0
EPS_TRUE = 50e-6


def sources(n):
    import librosa
    out = []
    for name in ("libri1", "libri2"):
        s, _ = librosa.load(librosa.ex(name), sr=FS, mono=True)
        s = s[:n]
        out.append(s / np.std(s))
    return out


def simulate(srcs):
    """2台の録音機（0.6 m 離す）での、目的と妨害の像（それぞれ (n, 2)）"""
    dim = np.array([6.0, 5.0, 3.0])
    e_abs, max_order = pra.inverse_sabine(0.2, dim)
    room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    room.add_microphone_array(np.array([[2.7, 3.3], [2.5, 2.5], [1.2, 1.2]]))
    room.add_source([3.5, 4.0, 1.5], signal=srcs[0])                  # 目的（左右対称にならない位置）
    room.add_source([4.6, 1.6, 1.5], signal=srcs[1])                  # 妨害
    pre = room.simulate(return_premix=True)                          # (音源, マイク, 時間)
    n = len(srcs[0])
    return pre[0, :, :n].T, pre[1, :, :n].T


def with_sfo(img, eps):
    """2台目（列1）だけ標本化周波数を fs(1+eps) にする"""
    out = img.copy()
    out[:, 1], used = ar.sfo_resample(img[:, 1], eps)
    return out, used


def coherence(X):
    R = ar.scm(X)
    return np.abs(R[:, 0, 1]) / np.sqrt(np.real(R[:, 0, 0] * R[:, 1, 1]) + 1e-20)


def main():
    print("[1] 式の数値例（fs = 16 kHz、L = 1024、S = 256）")
    for e in [10e-6, 50e-6, 100e-6]:
        for tsec in [10, 60]:
            print(f"  ε = {e*1e6:3.0f} ppm、{tsec:2d} 秒後：ずれ {e*tsec*1e3:5.2f} ms = {e*tsec*FS:5.1f} 標本、"
                  f"1 kHz の位相 {2*np.pi*1000*e*tsec:6.2f} rad")
        alpha = 2 * np.pi * 1000 * e * S / FS
        print(f"     1 フレームあたりの位相の回転（1 kHz）α = {alpha:.5f} rad")
    T10 = int(10 * FS / S)
    a = 2 * np.pi * 1000 * EPS_TRUE * S / FS
    print(f"  50 ppm、1 kHz、10 秒（T = {T10}）平均したときのコヒーレンスの理論値 "
          f"|sin(Tα/2)/(T sin(α/2))| = {abs(np.sin(T10*a/2)/(T10*np.sin(a/2))):.3f}")

    n = int(SEC * FS)
    tgt, itf = simulate(sources(n))
    tgt_e, used = with_sfo(tgt, EPS_TRUE)
    itf_e, _ = with_sfo(itf, EPS_TRUE)
    print(f"\n（使った ε = {used*1e6:.3f} ppm）")

    print("[2] GCC-PHAT を 1 秒ごとに（目的の音だけ）。2台目の遅れ [標本]")
    taus, tc = [], []
    for b in range(int(SEC)):
        seg = slice(b * FS, (b + 1) * FS)
        tau, _, _ = ar.gcc_phat(tgt_e[seg, 0], tgt_e[seg, 1], FS, max_tau=0.01, interp=16)
        taus.append(tau * FS)
        tc.append(b + 0.5)
    taus, tc = np.array(taus), np.array(tc)
    print("  " + "  ".join(f"{t:.2f}" for t in taus))
    fit = sm.OLS(taus, sm.add_constant(tc)).fit()
    slope, lo, hi = fit.params[1], *fit.conf_int()[1]
    print(f"  回帰の傾き {slope:.4f} 標本/秒 → ε̂ = {slope/FS*1e6:.1f} ppm（95% 区間 {lo/FS*1e6:.1f}〜{hi/FS*1e6:.1f}、"
          f"真 {used*1e6:.1f}）。切片 {fit.params[0]:.2f} 標本（場所による時間差）")

    print("[3] コヒーレンス（100 Hz〜4 kHz の平均）と固有値の比を、平均するフレームの数 T で比べる（目的の音だけ）")
    X0 = stft(tgt, L=L, S=S)
    Xe = stft(tgt_e, L=L, S=S)
    f = np.arange(L // 2 + 1) * FS / L
    band = (f >= 100) & (f <= 4000)
    for T in [40, 160, 320, 640]:
        c0 = coherence(X0[:, :T])[band].mean()
        ce = coherence(Xe[:, :T])[band].mean()
        al = 2 * np.pi * f[band] * used * S / FS
        th = np.mean(np.abs(np.sin(T * al / 2) / (T * np.sin(al / 2))) * coherence(X0[:, :T])[band])
        lam = np.linalg.eigvalsh(ar.scm(Xe[:, :T]))
        lam0 = np.linalg.eigvalsh(ar.scm(X0[:, :T]))
        print(f"  T = {T:3d}（{T*S/FS:4.1f} 秒）：ずれなし {c0:.3f}、50 ppm {ce:.3f}（理論の目安 {th:.3f}）。"
              f"λ₂/λ₁ の中央値 ずれなし {10*np.log10(np.median(lam0[band,0]/lam0[band,1])):6.1f} dB、"
              f"50 ppm {10*np.log10(np.median(lam[band,0]/lam[band,1])):6.1f} dB")

    print("[4] 最初の 2 秒で作った MVDR を使い続けたときの SIR の改善 [dB]（2 秒ごと）")
    def sir_blocks(tg, it, comp_eps=None):
        Xt, Xi = stft(tg, L=L, S=S), stft(it, L=L, S=S)
        if comp_eps is not None:                                      # 2台目の位相を戻す
            ph = ar.sfo_phase(Xt.shape[0], Xt.shape[1], L, S, comp_eps).conj()
            Xt[:, :, 1] *= ph
            Xi[:, :, 1] *= ph
        T2 = int(2 * FS / S)
        Rt, Ri = ar.scm(Xt[:, :T2]), ar.scm(Xi[:, :T2])
        a = np.linalg.eigh(Rt)[1][:, :, -1]                           # 目的のステアリングベクトル（最大の固有ベクトル）
        a = a / a[:, :1]
        w = ar.mvdr_weights(a, Ri, loading=1e-3)
        yt, yi = ar.apply_weights(w, Xt), ar.apply_weights(w, Xi)
        out = []
        for b in range(Xt.shape[1] // T2):
            sl = slice(b * T2, (b + 1) * T2)
            sir_in = 10 * np.log10(np.sum(np.abs(Xt[band, sl, 0]) ** 2) / np.sum(np.abs(Xi[band, sl, 0]) ** 2))
            sir_out = 10 * np.log10(np.sum(np.abs(yt[band, sl]) ** 2) / np.sum(np.abs(yi[band, sl]) ** 2))
            out.append(sir_out - sir_in)
        return np.array(out)
    res = {"ずれなし": sir_blocks(tgt, itf), "50 ppm": sir_blocks(tgt_e, itf_e),
           "50 ppm を補正": sir_blocks(tgt_e, itf_e, comp_eps=used)}
    for k, v in res.items():
        print(f"  {k:10s}：" + "  ".join(f"{x:5.1f}" for x in v))

    print("[5] ε を知らずに推定する：J(ε) = Σ_f log det R̂_f(ε)（位相を戻したあとの空間相関行列）を最小にする ε")
    Xm = stft(tgt_e + itf_e, L=L, S=S)
    grid = np.arange(-200, 200.5, 0.5) * 1e-6
    J = []
    for e in grid:
        Xc = Xm.copy()
        Xc[:, :, 1] *= ar.sfo_phase(Xc.shape[0], Xc.shape[1], L, S, e).conj()
        _, logdet = np.linalg.slogdet(ar.scm(Xc)[band])
        J.append(logdet.sum())
    J = np.array(J)
    print(f"  ε̂ = {grid[np.argmin(J)]*1e6:.1f} ppm（真 {used*1e6:.1f} ppm、格子 0.5 ppm、混合音（2音源）で）")

    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    axs[0].plot(tc, taus, "o", label="1 秒ごとの GCC-PHAT")
    axs[0].plot(tc, fit.fittedvalues, "-", label=f"回帰：{slope/FS*1e6:.1f} ppm")
    axs[0].set(xlabel="時刻 [s]", ylabel="2台目の遅れ [標本]", title="時間差が時間に比例して増える")
    for k, v in res.items():
        axs[1].plot(np.arange(len(v)) * 2 + 1, v, "o-", label=k)
    axs[1].set(xlabel="時刻 [s]", ylabel="SIR の改善 [dB]", title="最初に作った MVDR を使い続ける")
    axs[2].plot(grid * 1e6, J - J.min())
    axs[2].axvline(used * 1e6, color="k", ls="--", lw=1)
    axs[2].set(xlabel="補正に使う ε [ppm]", ylabel="J(ε) − 最小", title="Σ_f log det R̂_f(ε)")
    for a_ in axs:
        a_.grid(True, alpha=0.3)
    axs[0].legend(fontsize=8)
    axs[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("sfo.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("sfo.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"ε =  50 ppm、60 秒後：ずれ  [\d.]+ ms =\s+([\d.]+) 標本") == 48 and abs(num(r"コヒーレンスの理論値 .*? = ([\d.]+)") - 0.637) < 1e-3, "[1] 50 ppm・60 秒のずれが 48 標本、10 秒・1 kHz のコヒーレンスの理論値 0.637")
e2, lo, hi = (num(r"ε̂ = ([\d.]+) ppm（95% 区間 ([\d.]+)〜([\d.]+)、真", i) for i in (1, 2, 3))
ok(abs(e2 - 50) < 2 and lo <= 50 <= hi, f"[2] 傾きの推定 {e2} ppm（50 ppm 前後）、95% 区間 {lo}〜{hi} が真の値を含む")
co = re.findall(r"ずれなし ([\d.]+)、50 ppm ([\d.]+)（理論", cap.stdout)
c0, c50 = [float(a) for a, b in co], [float(b) for a, b in co]
ok(all(a > b for a, b in zip(c50, c50[1:])) and max(c0) - min(c0) < 0.02, f"[3] 50 ppm のコヒーレンスが平均の長さとともに下がる {c50}（ずれなしは下がらない {c0}）")
sir = lambda name: [float(v) for v in re.search(rf"{name}\s*：([\s\d.-]+)\n", cap.stdout).group(1).split()]
s50, s50c = sir("50 ppm  "), sir("50 ppm を補正")
ok(s50[0] - s50[-1] > 5 and min(s50c) > 7, f"[4] 50 ppm の SIR の改善が時間とともに下がり {s50}、補正すると戻る {s50c}")
ok(num(r"\[5\][\s\S]*?ε̂ = ([\d.]+) ppm") == 50.0, "[5] ε̂ が 50.0 ppm")

## 7. 拍手の時間差の解析を、合成の録音で試す（手順 I の 16、`clap_gcc.py --synthetic`）

実際に Mac とスマホで拍手を録る手順 I の 17〜22 は、このノートブックには入れていません（Mac のターミナルで行います）。ここでは、ずれ 40 ppm の合成の録音（`data/` に作られます）で、解析の流れが動くことを確かめます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['clap_gcc.py', '--synthetic']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-5：Mac とスマホの2台で拍手を録り、GCC-PHAT で時間差を求める（2台は同期していない）

使い方（~/home-lab/partIV/iv5 で）
  1) 自分の録音：uv run python clap_gcc.py data/mac.wav data/phone.wav
  2) 試し（合成の録音を作ってから解析）：uv run python clap_gcc.py --synthetic
やること
  [1] 2つの録音を 16 kHz にそろえる（スマホは 44.1 kHz や 48 kHz のことが多い）
  [2] 開始のずれ：最初の 20 秒どうしの GCC-PHAT（±5 秒の範囲）でおおまかにそろえる
  [3] Mac の録音から拍手を見つけ（包絡の山）、拍手ごとに ±20 ms の範囲で GCC-PHAT（16 倍に細かく）
  [4] 拍手ごとの時間差 τ_k を時刻 t_k に回帰する：傾き = 標本化周波数のずれ（ppm）、切片 = 開始のずれの残り＋場所による時間差
出力: clap_gcc.png、clap_gcc.csv（拍手ごとの時刻と時間差）
"""
import argparse
import pathlib
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.signal import resample_poly, find_peaks
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import plotting                                          # noqa: E402,F401
from common.audio_io import load, save                               # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000


def to_mono16k(path):
    x, fs = load(path)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if fs != FS:
        g = np.gcd(int(fs), FS)
        x = resample_poly(x, FS // g, int(fs) // g)
    return x, fs


def make_synthetic(folder, eps=40e-6, offset=1.234, sec=80.0, rng=None):
    """合成の録音：5 秒ごとに拍手（減衰する雑音のバースト）。スマホは 44.1 kHz で、時計が eps だけ速く、offset 秒早く録り始めた"""
    rng = rng or np.random.default_rng(1)
    fs0 = 48000
    n = int((sec + offset + 1) * fs0)
    base = np.zeros(n)
    for k in range(int(sec // 5)):
        i = int((2.0 + offset + 5 * k) * fs0)
        burst = rng.standard_normal(int(0.03 * fs0)) * np.exp(-np.arange(int(0.03 * fs0)) / (0.004 * fs0))
        base[i:i + len(burst)] += burst
    room_ir = np.zeros(int(0.2 * fs0))                                # 簡単な残響（指数減衰の雑音）
    room_ir[0] = 1.0
    room_ir[1:] = 0.15 * rng.standard_normal(len(room_ir) - 1) * np.exp(-np.arange(1, len(room_ir)) / (0.04 * fs0))
    wet = np.convolve(base, room_ir)[:n]
    mac = wet[int(offset * fs0):int((offset + sec) * fs0)]             # Mac は offset 秒あとから録り始めた
    phone = np.roll(wet, 30)                                          # スマホは拍手の場所から 30 標本（0.6 ms）遠い
    phone = resample_poly(phone, 147, 160)                            # 48 kHz → 44.1 kHz
    phone, used = ar.sfo_resample(phone, eps)
    phone = phone[: int((sec + offset) * 44100)]
    mac = mac + 0.003 * rng.standard_normal(len(mac))
    phone = phone + 0.003 * rng.standard_normal(len(phone))
    folder.mkdir(parents=True, exist_ok=True)
    for name, x, fs in (("syn_mac.wav", mac, fs0), ("syn_phone.wav", phone, 44100)):
        save(folder / name, x / np.max(np.abs(x)) * 0.5, fs)
    print(f"合成の録音を作りました：ずれ {used*1e6:.2f} ppm（スマホの時計が速い）、スマホが {offset} 秒早く録り始めた、"
          f"場所による遅れ 30/48000 s = {30/48000*1e3:.3f} ms")
    return folder / "syn_mac.wav", folder / "syn_phone.wav"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mac", nargs="?")
    ap.add_argument("phone", nargs="?")
    ap.add_argument("--synthetic", action="store_true")
    ap.add_argument("--thresh_db", type=float, default=-25.0, help="拍手とみなす包絡の高さ（最大からの dB）")
    a = ap.parse_args()
    if a.synthetic:
        a.mac, a.phone = make_synthetic(pathlib.Path("data"))
    if not (a.mac and a.phone):
        sys.exit("使い方: uv run python clap_gcc.py data/mac.wav data/phone.wav（または --synthetic）")
    x1, fs1 = to_mono16k(a.mac)
    x2, fs2 = to_mono16k(a.phone)
    print(f"[1] Mac {fs1} Hz・{len(x1)/FS:.1f} 秒、スマホ {fs2} Hz・{len(x2)/FS:.1f} 秒 → どちらも 16 kHz に")

    n20 = min(20 * FS, len(x1), len(x2))
    tau0, _, _ = ar.gcc_phat(x1[:n20], x2[:n20], FS, max_tau=5.0)
    shift = int(round(tau0 * FS))
    first = "スマホ" if tau0 > 0 else "Mac"
    print(f"[2] 開始のずれ：同じ音が、スマホのファイルでは Mac より {tau0:+.4f} 秒あとに現れる"
          f"（＝{first}が先に録り始めた。{shift:+d} 標本としてそろえる）")

    env = np.convolve(np.abs(x1), np.ones(160) / 160, mode="same")    # 10 ms の包絡
    env_db = 20 * np.log10(env / env.max() + 1e-12)
    peaks, _ = find_peaks(env_db, height=a.thresh_db, distance=FS)    # 1 秒以上離れた山
    rows = []
    for p in peaks:
        lo = p - int(0.05 * FS)
        hi = p + int(0.25 * FS)
        if lo < 0 or hi + shift > len(x2) or lo + shift < 0 or hi > len(x1):
            continue
        tau, lags, cc = ar.gcc_phat(x1[lo:hi], x2[lo + shift:hi + shift], FS, max_tau=0.02, interp=16)
        rows.append(dict(t=p / FS, tau_ms=(tau + shift / FS) * 1e3, peak=float(cc.max())))
    d = pd.DataFrame(rows)
    if len(d) < 3:
        sys.exit("拍手が3回以上見つかりません。--thresh_db を下げる（例 −30）か、拍手を大きく・回数を多くする")
    d.to_csv("clap_gcc.csv", index=False)
    print(f"[3] 拍手 {len(d)} 回：時間差 τ [ms] = " + "、".join(f"{v:.3f}" for v in d["tau_ms"]))

    fit = sm.OLS(d["tau_ms"], sm.add_constant(d["t"])).fit()
    b0, b1 = fit.params
    lo, hi = fit.conf_int().loc["t"]
    print(f"[4] 回帰：τ = {b0:.3f} ms + {b1*1e3:.2f} µs/秒 × t")
    print(f"    傾き → 標本化周波数のずれ ε̂ = {b1*1e3:.1f} ppm（95% 区間 {lo*1e3:.1f}〜{hi*1e3:.1f} ppm）。"
          f"残差の標準偏差 {np.std(fit.resid, ddof=2)*1e3:.1f} µs")
    print("    正の傾き：スマホの録音が時間とともに遅れていく（スマホの実際の標本化周波数の、名目の値に対する比が、"
          "Mac の比より ε だけ大きい）")
    print(f"    {len(x1)/FS:.0f} 秒の録音の最初と最後で、時間差は {b1*len(x1)/FS:.3f} ms 変わる"
          f"（5.66 cm のマイク間の最大の時間差 {0.0566/ar.C_SOUND*1e3:.3f} ms と比べる）")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(d["t"], d["tau_ms"], "o", label="拍手ごとの GCC-PHAT")
    ax.plot(d["t"], fit.fittedvalues, "-", label=f"回帰：{b1*1e3:.1f} ppm")
    ax.set(xlabel="Mac の録音の時刻 [s]", ylabel="スマホの遅れ τ [ms]", title="2台の時間差は時間とともに変わる")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig("clap_gcc.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("clap_gcc.png、clap_gcc.csv を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok("Mac より +1.2354 秒あとに現れる（＝スマホが先に録り始めた" in cap.stdout, "同じ音が、スマホのファイルでは Mac より +1.2354 秒あとに現れる（＝スマホが先に録り始めた）")
ok(num(r"拍手 (\d+) 回") == 16, "拍手 16 回")
ok(abs(num(r"ε̂ = ([\d.]+) ppm") - 40.0) < 0.1, "傾きの推定 40.0 ppm（真 40.00）")

## 8. 到来方向の推定誤差を、部屋を単位にして比べる（IV-5-3-6・3-11、手順 G、`doa_experiment.py`・`doa_stats.py`）

**数分〜10 分前後かかります**（執筆時の Mac で 2〜3 分。部屋 10 個 × 残響 3 × SNR 2 × くり返し 5。Colab ではもっとかかることがあります）。急ぐときは `sys.argv` の `"--seeds", "10"` を `"3"` にすると短くなりますが、そのときは下の確かめのうち行の数（3300 行）が合わなくなります。

`doa_experiment.py` は、`doa_results.csv` がすでにあると上書きしないように止まります。ノートブックを上から実行し直しても止まらないように、前の `doa_results.csv` を消してから実行します（残したいときは、先に名前を変えてください）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import pathlib
pathlib.Path("doa_results.csv").unlink(missing_ok=True)   # ノートブック版：前の結果を消す（上から実行し直しても止まらないように）
import sys
sys.argv = ['doa_experiment.py', '--seeds', '10', '--reps', '5', '--out', 'doa_results.csv']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-5：到来方向の推定誤差を、残響時間・SNR・マイク間隔で比べる（部屋のシミュレーション）

使い方: cd ~/home-lab/partIV/iv5 && uv run python doa_experiment.py --seeds 10 --reps 5 --out doa_results.csv
1つの「部屋」（シード）ごとに、部屋の大きさ・アレイの位置・音源の方向と距離・話し声の区間を乱数で決める。
同じ部屋に、次のアレイを同時に置く（中心は同じ）。
  直線（ULA）4本：間隔 d = 2.83・5.66・11.32 cm の3通り（x 軸に沿う。方向は 0〜180° で答える）
  円形（UCA）6本：半径 5.66 cm（方向は 0〜360° で答える）
残響時間 RT60 = 0（反射なし）・0.3・0.6 s（設定値）、SNR = 20・0 dB（マイクごとに独立な白色雑音）。
同じ部屋・同じ条件で、雑音だけを変えて reps 回くり返す（「同じ部屋のくり返し」は独立ではない。doa_stats.py で扱う）。
方法：GCC-PHAT（ULA の中央の2本）、MUSIC・SRP-PHAT（自前。ULA・UCA）。UCA では pra.doa の MUSIC・SRP の答えとも一致を確かめる。
結果は1行1推定：seed, rt60, snr, rep, array, method, true, est, err（度）
"""
import argparse
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import pyroomacoustics as pra

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common.stft import stft                                         # noqa: E402
from common import array as ar                                       # noqa: E402

FS = 16000
L, S = 512, 256
SEC = 3.0
F_LO, F_HI = 300.0, 4000.0
SPACINGS = [0.0283, 0.0566, 0.1132]
GRID_ULA = np.radians(np.arange(0, 180.01, 0.5))
GRID_UCA = np.radians(np.arange(0, 360, 1.0))


def speech_pool():
    import librosa
    s, _ = librosa.load(librosa.ex("libri1"), sr=FS, mono=True)
    return s


def make_room(seed, rt60, pool):
    """シードごとの部屋。rt60 だけを変えても、部屋の形と配置は同じにする（乱数をシードだけで決める）"""
    rng = np.random.default_rng(seed)
    dim = np.array([rng.uniform(5, 8), rng.uniform(4, 6), rng.uniform(2.7, 3.2)])
    center = np.array([dim[0] / 2 + rng.uniform(-0.6, 0.6), dim[1] / 2 + rng.uniform(-0.6, 0.6), 1.3])
    az = rng.uniform(np.radians(20), np.radians(160))                # 音源の方向（ULA の軸から）
    dist = rng.uniform(1.0, 2.0)
    off = rng.integers(0, len(pool) - int(SEC * FS))
    sig = pool[off:off + int(SEC * FS)]
    if rt60 == 0:
        room = pra.ShoeBox(dim, fs=FS, max_order=0)
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    arrays = {f"ULA{d*100:.2f}": ar.linear_array(4, d) for d in SPACINGS}
    arrays["UCA"] = ar.circular_array(6, 0.0566)
    allxy = np.hstack(list(arrays.values()))
    room.add_microphone_array(np.vstack([allxy + center[:2, None], np.full(allxy.shape[1], center[2])]))
    room.add_source(center + dist * np.array([np.cos(az), np.sin(az), 0.0]), signal=sig / np.std(sig))
    room.simulate()
    x = room.mic_array.signals[:, : len(sig)].T
    out, i = {}, 0
    for k, xy in arrays.items():
        out[k] = (xy, x[:, i:i + xy.shape[1]])
        i += xy.shape[1]
    return out, az


def ang_err(est, true, full_circle):
    d = np.degrees(abs(est - true))
    return min(d, 360 - d) if full_circle else d


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--reps", type=int, default=5)
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.0, 0.3, 0.6])
    ap.add_argument("--snr", type=float, nargs="+", default=[20.0, 0.0])
    ap.add_argument("--out", default="doa_results.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    pool = speech_pool()
    rows, mism = [], 0
    f = np.arange(L // 2 + 1) * FS / L
    bins = np.where((f >= F_LO) & (f < F_HI))[0]
    t0 = time.time()
    for seed in range(a.seeds):
        for rt60 in a.rt60:
            arrays, az = make_room(seed, rt60, pool)
            for snr in a.snr:
                for rep in range(a.reps):
                    rng = np.random.default_rng([seed, int(rt60 * 1000), int(snr), rep])
                    for name, (xy, x) in arrays.items():
                        p = np.mean(x[:, 0] ** 2)
                        xn = x + rng.standard_normal(x.shape) * np.sqrt(p / 10 ** (snr / 10))
                        X = stft(xn, L=L, S=S)[bins]
                        full = name == "UCA"
                        grid = GRID_UCA if full else GRID_ULA
                        est = {}
                        if not full:
                            d = float(name[3:]) / 100
                            # 中央の2本（1・2番）を使う：組の中心がアレイの中心と同じになり、真の方向とそろう
                            tau, _, _ = ar.gcc_phat(xn[:, 1], xn[:, 2], FS, max_tau=d / ar.C_SOUND * 1.05, interp=16)
                            est["GCC-PHAT"] = ar.tdoa_to_azimuth(tau, d)
                        est["MUSIC"] = grid[np.argmax(ar.music_spectrum(X, xy, f[bins], grid, 1))]
                        est["SRP-PHAT"] = grid[np.argmax(ar.srp_phat_spectrum(X, xy, f[bins], grid))]
                        if full and rep == 0:                         # pra.doa と同じ答えになるか
                            Xp = np.transpose(stft(xn, L=L, S=S), (2, 0, 1))
                            for m, cls in (("MUSIC", pra.doa.MUSIC), ("SRP-PHAT", pra.doa.SRP)):
                                dd = cls(xy, FS, L, c=ar.C_SOUND, azimuth=grid)
                                dd.locate_sources(Xp, freq_bins=bins)
                                mism += int(abs(dd.azimuth_recon[0] - est[m]) > 1e-9)
                        for m, e in est.items():
                            rows.append(dict(seed=seed, rt60=rt60, snr=snr, rep=rep, array=name, method=m,
                                             true=np.degrees(az), est=np.degrees(e), err=ang_err(e, az, full)))
        print(f"部屋 {seed} 終了（{time.time()-t0:.0f} 秒）")
        pd.DataFrame(rows).to_csv(a.out, index=False)
    print(f"{a.out} に保存しました（{len(rows)} 行）。UCA で pra.doa と答えが違った回数：{mism}")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(len(re.findall(r"部屋 \d+ 終了", cap.stdout)) == 10, "部屋ごとに「部屋 k 終了」と出る（10 個）")
ok("UCA で pra.doa と答えが違った回数：0" in cap.stdout, "最後に「UCA で pra.doa と答えが違った回数：0」")
ok(num(r"（(\d+) 行）") == 3300, "doa_results.csv が 3300 行")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['doa_stats.py', 'doa_results.csv']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-5：到来方向の推定誤差（doa_results.csv）を、部屋（シード）を単位にして集計・比較する

使い方: cd ~/home-lab/partIV/iv5 && uv run python doa_stats.py doa_results.csv
  [1] 条件ごとの誤差：部屋ごとの平均（くり返しで平均）→ 部屋10個の平均±標準偏差（音源分離の論文の作法）、
      全推定の中央値、10° を超えた割合（大外れ）
  [2] 統計の誤用：同じ部屋のくり返し（雑音だけ違う）を独立とみなして、2つの方法を比べる
      誤り：部屋 × くり返し = 50 組を独立な対として t 検定
      正しい：くり返しを部屋の中で平均し、部屋 10 個を対にして t 検定・ウィルコクソンの符号順位検定
      あわせて、級内相関 ICC と、有効なデータの数 n_eff = n / (1 + (m − 1)·ICC) を出す
  [3] 平均と中央値：大外れがあると平均は大きく引っぱられる
出力: doa_stats.png
"""
import sys

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def icc_oneway(y, groups):
    """一元配置の級内相関 ICC(1) = (MSB − MSW) / (MSB + (m − 1) MSW)（各群のくり返しの数 m は等しいとする）"""
    df = pd.DataFrame({"y": y, "g": groups})
    k = df["g"].nunique()
    m = len(df) // k
    gm = df.groupby("g")["y"].mean()
    msb = m * np.sum((gm - df["y"].mean()) ** 2) / (k - 1)
    msw = np.sum((df["y"] - df["g"].map(gm)) ** 2) / (len(df) - k)
    return (msb - msw) / (msb + (m - 1) * msw), m


def main(path):
    d = pd.read_csv(path)
    n_seed, n_rep = d["seed"].nunique(), d["rep"].nunique()
    print(f"部屋 {n_seed} 個 × くり返し {n_rep} 回")
    room = d.groupby(["array", "method", "rt60", "snr", "seed"], as_index=False)["err"].mean()
    print("\n[1] 推定誤差 [度]：部屋ごとの平均の、部屋をまたいだ平均±標準偏差（括弧は全推定の中央値、10°超の割合）")
    for (arr, met), g in room.groupby(["array", "method"]):
        cells = []
        for (rt, snr), gg in g.groupby(["rt60", "snr"]):
            allv = d[(d.array == arr) & (d.method == met) & (d.rt60 == rt) & (d.snr == snr)]["err"]
            cells.append(f"RT{rt:.1f}/{snr:.0f}dB {gg['err'].mean():5.1f}±{gg['err'].std(ddof=1):4.1f}"
                         f"（{allv.median():4.1f}、{(allv > 10).mean()*100:3.0f}%）")
        print(f"  {arr:9s} {met:8s} " + "  ".join(cells))

    print("\n[2] 統計の誤用：同じ部屋のくり返しを独立とみなす（ULA 5.66 cm、MUSIC − SRP-PHAT の誤差の差）")
    for rt, snr in [(0.3, 20.0), (0.3, 0.0)]:
        sub = d[(d.array == "ULA5.66") & (d.rt60 == rt) & (d.snr == snr)]
        p = sub.pivot_table(index=["seed", "rep"], columns="method", values="err")
        diff_all = (p["MUSIC"] - p["SRP-PHAT"]).to_numpy()
        diff_room = (p["MUSIC"] - p["SRP-PHAT"]).groupby(level="seed").mean().to_numpy()
        t_wrong = stats.ttest_1samp(diff_all, 0)
        t_right = stats.ttest_1samp(diff_room, 0)
        w_right = stats.wilcoxon(diff_room)
        ci_w = t_wrong.confidence_interval()
        ci_r = t_right.confidence_interval()
        icc, m = icc_oneway(diff_all, np.repeat(np.arange(n_seed), n_rep))
        n_eff = len(diff_all) / (1 + (m - 1) * icc)
        print(f"  RT60 {rt} s・SNR {snr:.0f} dB：差の平均 {diff_all.mean():+.2f}°")
        print(f"    【誤り】{len(diff_all)} 組を独立とみなす t 検定：p = {t_wrong.pvalue:.3g}、"
              f"95% 区間 {ci_w.low:+.2f}〜{ci_w.high:+.2f}")
        print(f"    【正しい】部屋 {len(diff_room)} 個の t 検定：p = {t_right.pvalue:.3g}、"
              f"95% 区間 {ci_r.low:+.2f}〜{ci_r.high:+.2f}。ウィルコクソン p = {w_right.pvalue:.3g}")
        print(f"    級内相関 ICC = {icc:.2f} → 有効なデータの数 n_eff ≈ {n_eff:.1f}（見かけは {len(diff_all)}）")

    print("\n[3] 平均と中央値（全条件・全方法をまとめた誤差）")
    print(f"  平均 {d['err'].mean():.1f}°、中央値 {d['err'].median():.1f}°、10° を超えた割合 {(d['err'] > 10).mean()*100:.0f}%")

    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for ax, snr in zip(axs, sorted(d["snr"].unique(), reverse=True)):
        labels, xs = [], 0
        for arr in ["ULA2.83", "ULA5.66", "ULA11.32", "UCA"]:
            for met in ["GCC-PHAT", "MUSIC", "SRP-PHAT"]:
                g = room[(room.array == arr) & (room.method == met) & (room.snr == snr)]
                if g.empty:
                    continue
                for j, rt in enumerate(sorted(g["rt60"].unique())):
                    v = g[g.rt60 == rt]["err"]
                    ax.plot(np.full(len(v), xs + 0.25 * (j - 1)), v, "o", ms=3, alpha=0.6, color=f"C{j}",
                            label=f"RT60 {rt} s" if xs == 0 else None)
                labels.append(f"{arr}\n{met}")
                xs += 1
        ax.set_xticks(range(len(labels)), labels, fontsize=7, rotation=60)
        ax.set(ylabel="部屋ごとの平均誤差 [度]", title=f"SNR {snr:.0f} dB（点は部屋1つ）")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("doa_stats.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("doa_stats.png を保存しました")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit("使い方: uv run python doa_stats.py doa_results.csv")
    main(sys.argv[1])

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
rows = [l for l in cap.stdout.split("[2]")[0].splitlines() if "RT0.0/20dB" in l]
m = lambda l, c: float(re.search(rf"{c}\s+([\d.]+)±", l).group(1))
share = lambda l, c: float(re.search(rf"{c}\s+[\d.]+±\s*[\d.]+（\s*[\d.]+、\s*(\d+)%）", l).group(1))
ok(len(rows) == 11 and all(m(l, "RT0.0/20dB") < 3 for l in rows), "[1] 反射なし・20 dB ではどの方法も誤差が数度以下（11 通り）")
ok(all(m(l, "RT0.6/20dB") > m(l, "RT0.0/20dB") and share(l, "RT0.6/0dB") >= share(l, "RT0.0/0dB") for l in rows), "[1] 残響が長いほど誤差と「10° 超の割合」が増える")
p = [float(v) for v in re.findall(r"t 検定：p = ([\d.e+-]+)", cap.stdout)]   # 20 dB の誤り・正しい、0 dB の誤り・正しい
icc = [float(v) for v in re.findall(r"ICC = ([\d.]+) →", cap.stdout)]
ok(p[0] < p[1] / 100, f"[2] RT60 0.3 s・20 dB：【誤り】の p = {p[0]:.3g} が【正しい】の p = {p[1]:.3g} より桁違いに小さい（ICC = {icc[0]}）")
ok(0.5 < p[2] / p[3] < 2, f"[2] RT60 0.3 s・0 dB：【誤り】の p = {p[2]:.3g} と【正しい】の p = {p[3]:.3g} がほとんど変わらない（ICC = {icc[1]}）")
ok(abs(icc[0] - 0.99) < 0.02 and icc[1] < 0.1, "[2] ICC が 0.99（雑音がほとんど効かない）と 0.01 前後（くり返しごとに雑音で変わる）")

## 9. 確認問題の答え合わせ（手順 J の 23、`quiz_check.py`）

確認問題（教科書の 7 節）は、**先に紙で解いてから**実行してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-5：確認問題の答えをコードで確かめる

使い方: cd ~/home-lab/partIV/iv5 && uv run python quiz_check.py
"""
import pathlib
import sys

import numpy as np

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common import array as ar                                       # noqa: E402

C, FS = ar.C_SOUND, 16000


def main():
    print("問1 ステアリングベクトル（d = 5.66 cm、θ = 60°、1 kHz、マイク1 を基準）")
    mics = ar.linear_array(2, 0.0566)
    a = ar.steering_vector(mics, [1000.0], azimuth=np.radians(60))[0]
    r = a / a[0]
    print(f"  時間差 {0.0566*np.cos(np.radians(60))/C*1e6:.2f} µs（マイク2 が早い）、a/a₁ = [1, {r[1]:.4f}]、位相 {np.angle(r[1]):.4f} rad")
    print(f"  θ = 90° なら a/a₁ = {ar.steering_vector(mics, [1000.0], azimuth=np.pi/2)[0][1] / ar.steering_vector(mics, [1000.0], azimuth=np.pi/2)[0][0]:.4f}")

    print("問2 空間エイリアシング")
    for d in (0.0566, 0.0283):
        print(f"  d = {d*100:.2f} cm：c/(2d) = {C/(2*d):.0f} Hz")
    print(f"  8 kHz まで折り返さない間隔 d ≤ c/(2·8000) = {C/16000*100:.2f} cm")

    print("問3 主ローブの幅（M = 8、d = 5.66 cm、正面）")
    lam = C / 2000
    print(f"  2 kHz：2·arcsin(λ/(Md)) = {2*np.degrees(np.arcsin(lam/(8*0.0566))):.1f}°、"
          f"ゼロが現れ始める周波数 c/(Md) = {C/(8*0.0566):.0f} Hz")

    print("問5 2本、a = [1, 1]（正面）、R_n = [[1, ρ], [ρ, 1]]")
    a = np.array([1.0, 1.0])
    for rho in (0.0, 0.5, 0.9, -0.5):
        Rn = np.array([[1, rho], [rho, 1]])
        w = np.linalg.solve(Rn, a)
        w = w / (a @ w)
        p = w @ Rn @ w
        print(f"  ρ = {rho:+.1f}：w = {np.round(w, 4)}、出力の雑音パワー {p:.3f}（{10*np.log10(p):+.2f} dB、1本は 0 dB）")

    print("問6 遠方場の目安 2D²/λ")
    for D, f in ((3 * 0.0566, 4000), (7 * 0.0566, 4000), (0.0566, 1000)):
        print(f"  D = {D*100:.1f} cm、{f} Hz：{2*D**2/(C/f):.3f} m")

    print("問7 GCC-PHAT の整数の遅れだけで表せる角度（d = 5.66 cm、16 kHz）")
    kmax = 0.0566 / C * FS
    print(f"  最大の遅れ d/c·fs = {kmax:.2f} 標本")
    for k in range(-2, 3):
        print(f"  k = {k:+d}：θ = {np.degrees(ar.tdoa_to_azimuth(k / FS, 0.0566)):.1f}°")

    print("問8 SFO（50 ppm、16 kHz、S = 256）")
    e = 50e-6
    print(f"  2 分間のずれ {e*120*FS:.0f} 標本 = {e*120*1e3:.0f} ms")
    for f in (1000, 2000):
        al = 2 * np.pi * f * e * 256 / FS
        T = 625
        print(f"  {f} Hz：α = {al:.5f} rad/フレーム、T = {T}（10 秒）でのコヒーレンスの倍率 "
              f"{abs(np.sin(T*al/2)/(T*np.sin(al/2))):.3f}")

    print("問9 残響の尾のエネルギー（指数減衰：60 dB 下がるのに RT60）")
    for rt in (0.3, 0.6):
        tau = rt * 20 / 60
        print(f"  RT60 {rt} s：エネルギーが −20 dB になる時刻 {tau*1000:.0f} ms = {tau*FS:.0f} 標本")

    print("問10 有効なデータの数 n_eff = n/(1 + (m − 1)ICC)")
    for icc in (0.8, 0.99, 0.01):
        print(f"  n = 50、m = 5、ICC = {icc}：n_eff = {50/(1+4*icc):.1f}")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok("時間差 82.51 µs" in cap.stdout and "0.8686+0.4955j" in cap.stdout, "問1 82.51 µs、a/a₁ = [1, 0.8686+0.4955j]")
ok(num(r"8 kHz まで折り返さない間隔 d ≤ c/\(2·8000\) = ([\d.]+) cm") == 2.14, "問2 8 kHz まで折り返さない間隔 2.14 cm")
ok(num(r"2 kHz：2·arcsin\(λ/\(Md\)\) = ([\d.]+)°") == 44.5, "問3 主ローブの幅 44.5°")
ok(abs(num(r"ρ = \+0\.5：.*?出力の雑音パワー ([\d.]+)") - 0.75) < 1e-9, "問5 ρ = 0.5 で出力の雑音パワー 0.750")
ok(abs(num(r"n = 50、m = 5、ICC = 0\.8：n_eff = ([\d.]+)") - 11.9) < 0.05, "問10 ICC = 0.8 で n_eff = 11.9")

## おわりに

- 自前の `common/array.py` が pyroomacoustics と一致すること、空間相関行列の固有値、ビームパターンと空間エイリアシング、遅延和・MVDR・MPDR、瞬時混合近似と窓の長さ、非同期録音のずれ、部屋を単位にした到来方向の推定誤差を、数値と音で確かめました。
- このノートブックに入れていないもの：Mac とスマホで拍手を録って解析する手順 I の 17〜22、第7章の `bss_experiment.py` を使う手順 F の 12、論文を読む手順 J の 24。Mac のターミナルで、教科書の手順どおりに行ってください。